In [1]:
import sys
import scenicplus

print(f"Python version: {sys.version}")
print(f"SCENIC+: {scenicplus.__version__}")

Python version: 3.11.4 | packaged by conda-forge | (main, Jun 10 2023, 18:08:41) [Clang 15.0.7 ]
SCENIC+: 1.0a2


In [1]:
import os
import pickle
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ---- Paths (adjust base dir if needed) ----
BASE = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus"
SCPLUS_PKL = os.path.join(BASE, "scplus_obj_geno_with_auc_june_final_REPAIRED.pkl")
CHECKPOINT_PKL = os.path.join(BASE, "checkpoint_with_auc_june_final_REPAIRED.pkl")
FIG_OUT = os.path.join(BASE, "scenicplus_further_analysis", "manuscript_figures")
os.makedirs(FIG_OUT, exist_ok=True)

print("Loading final SCENIC+ object...")
with open(SCPLUS_PKL, "rb") as f:
    scplus_obj = pickle.load(f)

print("Loading checkpoint (tf assignment, signatures)...")
with open(CHECKPOINT_PKL, "rb") as f:
    ckpt = pickle.load(f)

tf_assignment = ckpt.get("tfassignment", ckpt.get("tf_assignment"))
dfmulti = ckpt.get("dfmultifixed", ckpt.get("dfmulti"))

print("Object loaded:", type(scplus_obj))
print("Metadata shape:", scplus_obj.metadata_cell.shape)
print("Metadata columns:", scplus_obj.metadata_cell.columns.tolist())

Loading final SCENIC+ object...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Loading checkpoint (tf assignment, signatures)...
Object loaded: <class 'scenicplus.scenicplus_class.SCENICPLUS'>
Metadata shape: (2446, 3)
Metadata columns: ['ct_geno', 'genotype', 'cell_type']


In [2]:
# ---- Metadata ----
meta = scplus_obj.metadata_cell[["cell_type", "genotype"]].copy()
meta["cell_type"] = meta["cell_type"].astype(str)
meta["genotype"] = meta["genotype"].astype(str)

# ---- Expression matrix (cells x genes), standardize gene names ----
expr_df = scplus_obj.to_df("EXP").copy()
expr_df.columns = expr_df.columns.astype(str).str.capitalize()

# ---- Accessibility matrix: stored regions x cells, transpose so cells are rows ----
acc_df = scplus_obj.to_df("ACC").copy()
if acc_df.shape[0] != meta.shape[0]:
    print("Transposing accessibility matrix so cells are rows...")
    acc_df = acc_df.T

# ---- eRegulon AUC matrices (Gene-based and Region-based) ----
auc_multispec = scplus_obj.uns["eRegulon_AUC_multi_spec"]
auc_gene = auc_multispec["Gene_based"].copy()
auc_region = auc_multispec["Region_based"].copy()

print("Expression:", expr_df.shape, "| Accessibility (cells x regions):", acc_df.shape)
print("Gene-based AUC:", auc_gene.shape, "| Region-based AUC:", auc_region.shape)

# ---- Keep only cells present across all matrices ----
common_cells = sorted(set(meta.index) & set(auc_gene.index) & set(auc_region.index) & set(acc_df.index))
print(f"Common cells across all matrices: {len(common_cells)}")

meta = meta.loc[common_cells]
auc_gene = auc_gene.loc[common_cells]
auc_region = auc_region.loc[common_cells]
acc_df = acc_df.loc[common_cells]
expr_df = expr_df.reindex(common_cells).dropna(how="all")

gc.collect()

Transposing accessibility matrix so cells are rows...
Expression: (2446, 56884) | Accessibility (cells x regions): (2446, 746029)
Gene-based AUC: (2446, 85) | Region-based AUC: (2446, 85)
Common cells across all matrices: 2446


5

In [6]:
# # In[DIAG]: Diagnose eRegulon naming and dominant-TF assignment

# print("=== 1. Raw eRegulon column name samples ===")
# print(auc_gene.columns.tolist()[:15])
# print(f"... total {auc_gene.shape[1]} eRegulons")

# print("\n=== 2. TF name extraction check ===")
# sample_names = auc_gene.columns.tolist()[:15]
# for name in sample_names:
#     print(f"{name!r:40s} -> parsed TF: {extract_tf_name(name)!r}")

# print("\n=== 3. Are eRegulon names unique per TF, or do multiple eRegulons share the same TF? ===")
# tf_counts = pd.Series([extract_tf_name(c) for c in auc_gene.columns]).value_counts()
# print(f"Unique TFs parsed: {tf_counts.shape[0]} from {auc_gene.shape[1]} eRegulons")
# print(tf_counts.head(10))

# print("\n=== 4. Does dominant eRegulon vary within a cell type, or collapse to one TF per cluster? ===")
# dominant_eregulon = auc_gene.idxmax(axis=1)
# dominant_tf = dominant_eregulon.map(lambda x: extract_tf_name(x))
# check_df = pd.DataFrame({
#     "cell_type": meta.loc[auc_gene.index, "cell_type"].values,
#     "dominant_TF": dominant_tf.values,
#     "dominant_eRegulon": dominant_eregulon.values,
# })
# crosstab = pd.crosstab(check_df["cell_type"], check_df["dominant_TF"])
# print(crosstab)

# print("\n=== 5. How many distinct dominant TFs appear per cell type? ===")
# print(check_df.groupby("cell_type")["dominant_TF"].nunique())

# print("\n=== 6. Overall: does dominant_TF == cell_type identity effectively (1 TF per cluster)? ===")
# n_dominant_tfs_total = check_df["dominant_TF"].nunique()
# n_cell_types = check_df["cell_type"].nunique()
# print(f"Total unique dominant TFs across all cells: {n_dominant_tfs_total}")
# print(f"Total cell types: {n_cell_types}")
# if n_dominant_tfs_total <= n_cell_types + 2:
#     print(">>> WARNING: dominant TF count is close to cell type count — "
#           "the 'idxmax per cell' approach is likely just re-deriving cluster identity, "
#           "not showing TF diversity within clusters.")

=== 1. Raw eRegulon column name samples ===
['Astrocytes_pos_manual_Gli3', 'Astrocytes_pos_manual_Gpam', 'Astrocytes_pos_manual_Msi1', 'Astrocytes_pos_manual_Prdm16', 'Astrocytes_pos_manual_Rfx4', 'Deep-layer extratelencephalic neurons_pos_manual_Ckmt1', 'Deep-layer extratelencephalic neurons_pos_manual_Fezf2', 'Deep-layer extratelencephalic neurons_pos_manual_Neurod1', 'Deep-layer extratelencephalic neurons_pos_manual_Nr4a1', 'Deep-layer extratelencephalic neurons_pos_manual_Otx1', 'Endothelial cells_pos_manual_Lef1', 'Endothelial cells_pos_manual_Mecom', 'Endothelial cells_pos_manual_Sox18', 'Endothelial cells_pos_manual_Tbx1', 'Endothelial cells_pos_manual_Zfp366']
... total 85 eRegulons

=== 2. TF name extraction check ===
'Astrocytes_pos_manual_Gli3'             -> parsed TF: 'Astrocytes'
'Astrocytes_pos_manual_Gpam'             -> parsed TF: 'Astrocytes'
'Astrocytes_pos_manual_Msi1'             -> parsed TF: 'Astrocytes'
'Astrocytes_pos_manual_Prdm16'           -> parsed TF: 'Ast

In [9]:
# # In[DIAG2]: Why are some cell types missing / showing multiple eRegulons in dominant-label UMAP?

# print("=== A. Which cell types have eRegulons defined at all? ===")
# eregulon_celltypes = sorted(set(ct for ct, tf in eregulon_info.values()))
# print(f"Cell types with eRegulons in auc_gene: {len(eregulon_celltypes)}")
# print(eregulon_celltypes)

# missing_from_eregulons = set(celltype_order) - set(eregulon_celltypes)
# print(f"\nCell types in celltype_order but with NO eRegulons defined: {missing_from_eregulons}")

# print("\n=== B. Which cell types actually appear in the dominant_label legend? ===")
# labels_present = adata_ereg.obs["dominant_label"].cat.categories.tolist()
# celltypes_in_legend = sorted(set(lbl.split(": ")[0] for lbl in labels_present))
# print(f"Cell types appearing as dominant somewhere: {len(celltypes_in_legend)}")
# print(celltypes_in_legend)

# never_dominant = set(eregulon_celltypes) - set(celltypes_in_legend)
# print(f"\nCell types that HAVE eRegulons but NEVER win the per-cell argmax: {never_dominant}")

# print("\n=== C. For cells whose own cell_type never wins, who steals dominance? ===")
# diag_df2 = pd.DataFrame({
#     "cell_type": meta.loc[auc_gene.index, "cell_type"].values,
#     "dominant_label": dominant_label.values,
#     "dominant_source_celltype": [lbl.split(": ")[0] for lbl in dominant_label.values],
# })
# for ct in never_dominant:
#     sub = diag_df2[diag_df2["cell_type"] == ct]
#     if len(sub) == 0:
#         print(f"{ct}: 0 cells of this type present in auc_gene index")
#         continue
#     steal_counts = sub["dominant_source_celltype"].value_counts()
#     print(f"\n{ct} (n={len(sub)} cells) -- dominant label actually comes from:")
#     print(steal_counts)

# print("\n=== D. How many distinct dominant labels per cell type (confirms 'multiple eRegulons per type') ===")
# print(diag_df2.groupby("cell_type")["dominant_label"].nunique().sort_values(ascending=False))

# print("\n=== E. Raw AUC scale check -- are some cell types' eRegulons systematically lower AUC? ===")
# mean_max_auc_by_ct = {}
# for ct in eregulon_celltypes:
#     ct_cols = [reg for reg, (c, t) in eregulon_info.items() if c == ct]
#     mean_max_auc_by_ct[ct] = auc_gene[ct_cols].max(axis=1).mean()
# print(pd.Series(mean_max_auc_by_ct).sort_values())

=== A. Which cell types have eRegulons defined at all? ===
Cell types with eRegulons in auc_gene: 17
['Astrocytes', 'Deep-layer extratelencephalic neurons', 'Endothelial cells', 'Layer 2/3 IT neurons', 'Layer 4 sensory neurons', 'Layer 5/6 IT neurons', 'Layer 5a IT neurons', 'Layer 5b PT neurons', 'Layer 6a corticothalamic neurons', 'Layer 6b neurons', 'Meningeal fibroblasts', 'Microglia', 'Oligodendrocyte precursor cells', 'Oligodendrocytes', 'PV+ interneurons', 'SST+ interneurons', 'VIP+ interneurons']

Cell types in celltype_order but with NO eRegulons defined: {'Myelinating oligodendrocytes', 'Leptomeningeal cells', 'Corticospinal neurons Type II', 'Perineuronal oligodendrocytes', 'Corticospinal neurons Type I'}

=== B. Which cell types actually appear in the dominant_label legend? ===
Cell types appearing as dominant somewhere: 14
['Astrocytes', 'Endothelial cells', 'Layer 4 sensory neurons', 'Layer 5/6 IT neurons', 'Layer 5a IT neurons', 'Layer 5b PT neurons', 'Layer 6a corticoth

In [3]:
adata_ereg = ad.AnnData(X=auc_gene.values.astype(float))
adata_ereg.obs_names = auc_gene.index
adata_ereg.var_names = auc_gene.columns
adata_ereg.obs["cell_type"] = meta["cell_type"].values
adata_ereg.obs["genotype"] = meta["genotype"].values

sc.pp.scale(adata_ereg, max_value=10)
sc.pp.pca(adata_ereg, n_comps=min(30, adata_ereg.shape[1] - 1))
sc.pp.neighbors(adata_ereg, n_neighbors=15)
sc.tl.umap(adata_ereg, random_state=42)

# fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
# sc.pl.umap(adata_ereg, color="cell_type", ax=axes[0], show=False, size=25, title="eRegulon UMAP — Cell type")
# sc.pl.umap(adata_ereg, color="genotype", ax=axes[1], show=False, size=25, title="eRegulon UMAP — Genotype",
#            palette={"Ctrl": "#2166ac", "KO": "#b2182b"})
# plt.tight_layout()
# outpath = os.path.join(FIG_OUT, "Fig1_eRegulon_AUC_UMAP.pdf")
# fig.savefig(outpath, bbox_inches="tight")
# plt.close(fig)
# print("Saved:", outpath)

In [63]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scanpy as sc
import os

# ---- Full reference palette (22 categories), same source as heatmap/UMAP figures ----
celltype_order = [
    "Layer 2/3 IT neurons", "Layer 4 sensory neurons", "Layer 5a IT neurons",
    "Layer 5b PT neurons", "Layer 5/6 IT neurons", "Layer 6a corticothalamic neurons",
    "Layer 6b neurons", "Deep-layer extratelencephalic neurons",
    "Corticospinal neurons Type I", "Corticospinal neurons Type II",
    "PV+ interneurons", "SST+ interneurons", "VIP+ interneurons",
    "Astrocytes", "Oligodendrocyte precursor cells", "Newly formed oligodendrocytes",
    "Perineuronal oligodendrocytes", "Myelinating oligodendrocytes",
    "Microglia", "Endothelial cells", "Leptomeningeal cells", "Meningeal fibroblasts"
]
hex_colors = [
    "6B5B95", "45B8AC", "955251", "4E84C4", "B565A7", "88B04B", "7B6888", "C3447A",
    "009B77", "EFC050", "7FCDCD", "DD4124", "5B5EA6", "E07A5F", "4BACC6", "E8A0BF",
    "ea8a33", "9B2335", "C17BAE", "DECF3F", "789262", "BC243C"
]

atac_colors = {ct: f"#{h}" for ct, h in zip(celltype_order, hex_colors)}

# ---- Collapse rule: any OL subtype variant (Newly formed / Perineuronal / Myelinating,
# or a pre-merged "Oligodendrocytes" label) must render as NFOL's original color ----
NFOL_COLOR = atac_colors["Newly formed oligodendrocytes"]  # "#E8A0BF"
for ol_label in ["Perineuronal oligodendrocytes", "Myelinating oligodendrocytes", "Oligodendrocytes"]:
    atac_colors[ol_label] = NFOL_COLOR

# ---- Restrict palette to cell types actually present in this object ----
present_cts = adata_ereg.obs["cell_type"].unique().tolist()
celltype_palette = {ct: atac_colors.get(ct, "#999999") for ct in present_cts}

unmatched = [ct for ct in present_cts if ct not in atac_colors]
if unmatched:
    print("Unmatched cell types (will render gray):", unmatched)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
sc.pl.umap(adata_ereg, color="cell_type", ax=axes[0], show=False, size=25,
           title="eRegulon UMAP — Cell type", palette=celltype_palette)
sc.pl.umap(adata_ereg, color="genotype", ax=axes[1], show=False, size=25,
           title="eRegulon UMAP — Genotype",
           palette={"Ctrl": "#2166ac", "KO": "#b2182b"})
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig1_eRegulon_AUC_UMAP.pdf")
fig.savefig(outpath, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig1_eRegulon_AUC_UMAP.pdf


/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [10]:
# In[NEW]: eRegulon UMAP colored by dominant driving TF (z-scored), labeled "CellType: TF", using atac_colors palette

import matplotlib.colors as mcolors
import colorsys

def parse_eregulon_name(eregulon_name):
    """eRegulon names follow '{cell_type}_pos_manual_{TF}' (e.g. 'Astrocytes_pos_manual_Gli3')."""
    name = str(eregulon_name)
    if "_pos_manual_" in name:
        cell_type_part, tf_part = name.split("_pos_manual_", 1)
    else:
        cell_type_part, tf_part = name, name
    return cell_type_part, tf_part

eregulon_info = {reg: parse_eregulon_name(reg) for reg in auc_gene.columns}
eregulon_to_label = {reg: f"{ct}: {tf}" for reg, (ct, tf) in eregulon_info.items()}

# ---- Use the z-scored matrix (adata_ereg.X, already scaled per eRegulon) instead of raw AUC ----
# This removes the systematic scale bias identified across cell types (raw mean-max-AUC ranged
# 0.06-0.31 across cell types), so each eRegulon's RELATIVE activation is compared, not its raw magnitude.
scaled_auc = pd.DataFrame(adata_ereg.X, index=adata_ereg.obs_names, columns=adata_ereg.var_names)

dominant_eregulon = scaled_auc.idxmax(axis=1)
dominant_label = dominant_eregulon.map(eregulon_to_label)

adata_ereg.obs["dominant_eRegulon"] = dominant_eregulon.reindex(adata_ereg.obs_names).values
adata_ereg.obs["dominant_label"] = dominant_label.reindex(adata_ereg.obs_names).values
adata_ereg.obs["dominant_label"] = adata_ereg.obs["dominant_label"].astype("category")

unique_labels = sorted(adata_ereg.obs["dominant_label"].cat.categories.tolist())
n_labels = len(unique_labels)

diag_df = pd.DataFrame({
    "cell_type": meta.loc[auc_gene.index, "cell_type"].values,
    "dominant_label": dominant_label.values,
    "dominant_source_celltype": [lbl.split(": ")[0] for lbl in dominant_label.values],
})
print("Cell types recovered as dominant after z-score fix:")
print(sorted(diag_df["dominant_source_celltype"].unique()))
print("\nCell types whose own regulon still never wins (should shrink vs raw-AUC version):")
print(set(diag_df["cell_type"].unique()) - set(diag_df["dominant_source_celltype"].unique()))
print("\nDistinct dominant labels per cell type:")
print(diag_df.groupby("cell_type")["dominant_label"].nunique().sort_values(ascending=False))

# ---- Build palette: base color from atac_colors, shaded per TF within same cell type ----
def shade_color(hex_color, factor):
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l_new = min(max(l * (0.55 + factor), 0.05), 0.92)
    r2, g2, b2 = colorsys.hls_to_rgb(h, l_new, s)
    return mcolors.to_hex((r2, g2, b2))

label_palette = {}
for ct in celltype_order:
    ct_labels = sorted([lbl for lbl in unique_labels if lbl.startswith(f"{ct}: ")])
    base_color = atac_colors.get(ct, "#999999")
    n_ct = len(ct_labels)
    for i, lbl in enumerate(ct_labels):
        factor = 0.5 if n_ct == 1 else 0.15 + (1.5 * i / (n_ct - 1))
        label_palette[lbl] = shade_color(base_color, factor)

for lbl in unique_labels:
    if lbl not in label_palette:
        label_palette[lbl] = "#999999"

fig, ax = plt.subplots(figsize=(11, 7.5))
sc.pl.umap(
    adata_ereg,
    color="dominant_label",
    ax=ax,
    show=False,
    size=25,
    palette=label_palette,
    title="eRegulon UMAP — Dominant driving TF (z-scored, Cell type: TF)",
    legend_loc="right margin",
    legend_fontsize=6,
    legend_fontweight="normal",
)
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig1d_eRegulon_AUC_UMAP_TF_legend.pdf")
fig.savefig(outpath, bbox_inches="tight", dpi=300)
plt.close(fig)
print(f"Saved: {outpath} | {n_labels} unique 'CellType: TF' labels shown in legend")

Cell types recovered as dominant after z-score fix:
['Astrocytes', 'Deep-layer extratelencephalic neurons', 'Endothelial cells', 'Layer 2/3 IT neurons', 'Layer 4 sensory neurons', 'Layer 5/6 IT neurons', 'Layer 5a IT neurons', 'Layer 5b PT neurons', 'Layer 6a corticothalamic neurons', 'Layer 6b neurons', 'Meningeal fibroblasts', 'Microglia', 'Oligodendrocyte precursor cells', 'Oligodendrocytes', 'PV+ interneurons', 'SST+ interneurons', 'VIP+ interneurons']

Cell types whose own regulon still never wins (should shrink vs raw-AUC version):
set()

Distinct dominant labels per cell type:
cell_type
Layer 5a IT neurons                      12
Layer 4 sensory neurons                   9
SST+ interneurons                         9
Astrocytes                                6
Layer 5/6 IT neurons                      6
Oligodendrocytes                          6
Microglia                                 5
Layer 2/3 IT neurons                      5
Meningeal fibroblasts                     5
End

/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig1d_eRegulon_AUC_UMAP_TF_legend.pdf | 67 unique 'CellType: TF' labels shown in legend


In [7]:
def compute_rss(auc_df, labels):
    """Jensen-Shannon-based regulon specificity score per eRegulon per cell type."""
    from scipy.spatial.distance import jensenshannon
    cell_types = sorted(labels.unique())
    rss = pd.DataFrame(index=auc_df.columns, columns=cell_types, dtype=float)
    for ct in cell_types:
        ct_indicator = (labels == ct).astype(float)
        ct_indicator = ct_indicator / ct_indicator.sum()
        for reg in auc_df.columns:
            vals = auc_df[reg].values
            if vals.sum() == 0:
                rss.loc[reg, ct] = 0.0
                continue
            vals_norm = vals / vals.sum()
            jsd = jensenshannon(vals_norm, ct_indicator.values)
            rss.loc[reg, ct] = 1 - jsd
    return rss

rss_df = compute_rss(auc_gene, meta["cell_type"])

TOP_N_RSS = 8
top_per_ct = {}
for ct in rss_df.columns:
    top_per_ct[ct] = rss_df[ct].sort_values(ascending=False).head(TOP_N_RSS)

fig, axes = plt.subplots(1, len(rss_df.columns), figsize=(3.2 * len(rss_df.columns), 5), sharey=False)
if len(rss_df.columns) == 1:
    axes = [axes]
for ax, ct in zip(axes, rss_df.columns):
    s = top_per_ct[ct]
    ax.barh(range(len(s)), s.values[::-1], color="#4393c3")
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels(s.index[::-1], fontsize=8)
    ax.set_title(ct, fontsize=10)
    ax.set_xlabel("RSS")
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig2_RSS_top_eRegulons_per_celltype.pdf")
fig.savefig(outpath, bbox_inches="tight")
plt.close(fig)

rss_df.to_csv(os.path.join(FIG_OUT, "RSS_matrix_all_eRegulons.csv"))
print("Saved:", outpath)

/var/folders/c5/9gnt7c7x7x59ygrp90w3n60r0000gn/T/ipykernel_29570/1051807186.py:36: UserWarning: Tight layout not applied. tight_layout cannot make axes width small enough to accommodate all axes decorations
  plt.tight_layout()


Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig2_RSS_top_eRegulons_per_celltype.pdf


In [16]:
adata_ereg_region = ad.AnnData(X=auc_region.values.astype(float))
adata_ereg_region.obs_names = auc_region.index
adata_ereg_region.var_names = auc_region.columns
adata_ereg_region.obs["cell_type"] = meta.loc[auc_region.index, "cell_type"].values
adata_ereg_region.obs["genotype"] = meta.loc[auc_region.index, "genotype"].values

sc.pp.scale(adata_ereg_region, max_value=10)
sc.pp.pca(adata_ereg_region, n_comps=min(30, adata_ereg_region.shape[1] - 1))
sc.pp.neighbors(adata_ereg_region, n_neighbors=15)
sc.tl.umap(adata_ereg_region, random_state=42)

present_cts_region = adata_ereg_region.obs["cell_type"].unique().tolist()
celltype_palette_region = {ct: atac_colors.get(ct, "#999999") for ct in present_cts_region}

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
sc.pl.umap(adata_ereg_region, color="cell_type", ax=axes[0], show=False, size=25,
           title="Region-based eRegulon UMAP — Cell type", palette=celltype_palette_region)
sc.pl.umap(adata_ereg_region, color="genotype", ax=axes[1], show=False, size=25,
           title="Region-based eRegulon UMAP — Genotype",
           palette={"Ctrl": "#2166ac", "KO": "#b2182b"})
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig1b_eRegulon_Region_AUC_UMAP.pdf")
fig.savefig(outpath, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig1b_eRegulon_Region_AUC_UMAP.pdf


/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [18]:
def top_variable_eregulons(auc_df, n_top=None, frac=0.5):
    variances = auc_df.var(axis=0).sort_values(ascending=False)
    if n_top is None:
        n_top = max(20, int(len(variances) * frac))
    return variances.head(n_top).index.tolist()

top_eregs_gene = top_variable_eregulons(auc_gene, frac=0.7)
top_eregs_region = top_variable_eregulons(auc_region, frac=0.7)

print(f"Using {len(top_eregs_gene)} / {auc_gene.shape[1]} gene-based eRegulons")
print(f"Using {len(top_eregs_region)} / {auc_region.shape[1]} region-based eRegulons")

adata_ereg_hv = ad.AnnData(X=auc_region[top_eregs_region].values.astype(float))
adata_ereg_hv.obs_names = auc_region.index
adata_ereg_hv.var_names = top_eregs_region
adata_ereg_hv.obs["cell_type"] = meta.loc[auc_region.index, "cell_type"].values
adata_ereg_hv.obs["genotype"] = meta.loc[auc_region.index, "genotype"].values

sc.pp.scale(adata_ereg_hv, max_value=10)
sc.pp.pca(adata_ereg_hv, n_comps=min(50, adata_ereg_hv.shape[1] - 1))
sc.pp.neighbors(adata_ereg_hv, n_neighbors=10, n_pcs=30)
sc.tl.umap(adata_ereg_hv, random_state=42, min_dist=0.2, spread=1.2)

present_cts_hv = adata_ereg_hv.obs["cell_type"].unique().tolist()
celltype_palette_hv = {ct: atac_colors.get(ct, "#999999") for ct in present_cts_hv}

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
sc.pl.umap(adata_ereg_hv, color="cell_type", ax=axes[0], show=False, size=25,
           title="Region-based eRegulon UMAP (top variable, tighter)", palette=celltype_palette_hv)
sc.pl.umap(adata_ereg_hv, color="genotype", ax=axes[1], show=False, size=25,
           title="Region-based eRegulon UMAP — Genotype",
           palette={"Ctrl": "#2166ac", "KO": "#b2182b"})
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig1c_eRegulon_Region_AUC_UMAP_topvariable.pdf")
fig.savefig(outpath, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath)

Using 59 / 85 gene-based eRegulons
Using 59 / 85 region-based eRegulons
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig1c_eRegulon_Region_AUC_UMAP_topvariable.pdf


/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [19]:
l23_cells = meta.index[meta["cell_type"] == "Layer 2/3 IT neurons"]
l5a_cells = meta.index[meta["cell_type"] == "Layer 5a IT neurons"]

l23_common = [c for c in l23_cells if c in auc_region.index]
l5a_common = [c for c in l5a_cells if c in auc_region.index]

diff_scores = []
for reg in auc_region.columns:
    v23 = auc_region.loc[l23_common, reg].values
    v5a = auc_region.loc[l5a_common, reg].values
    if v23.std() == 0 and v5a.std() == 0:
        continue
    try:
        _, p = mannwhitneyu(v23, v5a, alternative="two-sided")
    except ValueError:
        p = 1.0
    delta = v23.mean() - v5a.mean()
    diff_scores.append((reg, delta, p))

layer_diff_df = pd.DataFrame(diff_scores, columns=["eRegulon", "delta_AUC_L23_minus_L5a", "pval"])
_, layer_diff_df["padj"], _, _ = multipletests(layer_diff_df["pval"], method="fdr_bh")
layer_diff_df = layer_diff_df.sort_values("delta_AUC_L23_minus_L5a")

layer_diff_df.to_csv(os.path.join(FIG_OUT, "L23_vs_L5a_discriminating_eRegulons.csv"), index=False)
print(layer_diff_df[layer_diff_df["padj"] < 0.05].shape[0], "significantly discriminating eRegulons found")
print(layer_diff_df.head(10))
print(layer_diff_df.tail(10))

78 significantly discriminating eRegulons found
                                                 eRegulon  \
9   Deep-layer extratelencephalic neurons_pos_manual_Otx1   
33                 Layer 5a IT neurons_pos_manual_Neurod1   
60        Oligodendrocyte precursor cells_pos_manual_E2f8   
44     Layer 6a corticothalamic neurons_pos_manual_Zfp831   
61        Oligodendrocyte precursor cells_pos_manual_Msi1   
76                      SST+ interneurons_pos_manual_Dlx2   
64       Oligodendrocyte precursor cells_pos_manual_Traf4   
82                      VIP+ interneurons_pos_manual_Dlx5   
72                       PV+ interneurons_pos_manual_Dlx5   
84                    VIP+ interneurons_pos_manual_Zfp536   

    delta_AUC_L23_minus_L5a          pval          padj  
9                 -0.001817  2.499447e-15  1.118334e-14  
33                -0.001681  1.348596e-12  3.016597e-12  
60                -0.001680  2.313669e-15  1.118334e-14  
44                -0.001429  2.113260e-12  4.605

In [21]:
adata_atac = ad.AnnData(X=acc_df.values.astype(np.float32))
adata_atac.obs_names = acc_df.index
adata_atac.var_names = acc_df.columns
adata_atac.obs["cell_type"] = meta.loc[acc_df.index, "cell_type"].values
adata_atac.obs["genotype"] = meta.loc[acc_df.index, "genotype"].values

sc.pp.filter_genes(adata_atac, min_cells=int(0.01 * adata_atac.n_obs))
sc.pp.normalize_total(adata_atac, target_sum=1e4)
sc.pp.log1p(adata_atac)
sc.pp.highly_variable_genes(adata_atac, n_top_genes=min(20000, adata_atac.n_vars))
adata_atac = adata_atac[:, adata_atac.var.highly_variable].copy()

sc.pp.scale(adata_atac, max_value=10)
sc.pp.pca(adata_atac, n_comps=30)
sc.pp.neighbors(adata_atac, n_neighbors=15)
sc.tl.umap(adata_atac, random_state=42)

present_cts_atac = adata_atac.obs["cell_type"].unique().tolist()
celltype_palette_atac = {ct: atac_colors.get(ct, "#999999") for ct in present_cts_atac}

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
sc.pl.umap(adata_atac, color="cell_type", ax=axes[0], show=False, size=15,
           title="Accessibility UMAP — Cell type", palette=celltype_palette_atac)
sc.pl.umap(adata_atac, color="genotype", ax=axes[1], show=False, size=15,
           title="Accessibility UMAP — Genotype",
           palette={"Ctrl": "#2166ac", "KO": "#b2182b"})
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig3_chromatin_accessibility_UMAP.pdf")
fig.savefig(outpath, bbox_inches="tight")
plt.close(fig)
print("Saved:", outpath)

/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig3_chromatin_accessibility_UMAP.pdf


/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/Users/cnbr/miniforge3/envs/scenicplus_v2/lib/python3.11/site-packages/scanpy/plotting/_tools/scatterplots.py:378: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [ ]:
# MIN_MEAN_ACC = 0.05
# MIN_METACELLS = 10
# TOP_N_PER_CT = 30

# results = []
# for ct in meta["cell_type"].unique():
#     ct_mask = meta["cell_type"] == ct
#     ko_cells = meta.index[ct_mask & (meta["genotype"] == "KO")]
#     ctrl_cells = meta.index[ct_mask & (meta["genotype"] == "Ctrl")]
#     if len(ko_cells) < MIN_METACELLS or len(ctrl_cells) < MIN_METACELLS:
#         print(f"Skipping {ct}: KO={len(ko_cells)}, Ctrl={len(ctrl_cells)} metacells")
#         continue

#     ko_mean_all = acc_df.loc[ko_cells].mean(axis=0)
#     ctrl_mean_all = acc_df.loc[ctrl_cells].mean(axis=0)
#     keep = (ko_mean_all > MIN_MEAN_ACC) | (ctrl_mean_all > MIN_MEAN_ACC)
#     var_regions = acc_df.columns[keep]
#     if len(var_regions) == 0:
#         continue

#     ko_mat = acc_df.loc[ko_cells, var_regions].values
#     ctrl_mat = acc_df.loc[ctrl_cells, var_regions].values

#     log2fc = np.log2(ko_mat.mean(axis=0) + 1e-3) - np.log2(ctrl_mat.mean(axis=0) + 1e-3)

#     n1, n2 = ko_mat.shape[0], ctrl_mat.shape[0]
#     effect_size = np.zeros(len(var_regions))
#     for i in range(len(var_regions)):
#         try:
#             u_stat, _ = mannwhitneyu(ko_mat[:, i], ctrl_mat[:, i], alternative="two-sided")
#             effect_size[i] = (2 * u_stat / (n1 * n2)) - 1
#         except ValueError:
#             effect_size[i] = 0.0

#     res = pd.DataFrame({"region": var_regions, "log2FC": log2fc, "effect_size": effect_size,
#                          "cell_type": ct, "n_KO_metacells": n1, "n_Ctrl_metacells": n2})
#     results.append(res)

# diff_acc = pd.concat(results, ignore_index=True)
# diff_acc["log2FC_clipped"] = diff_acc["log2FC"].clip(-10, 10)
# diff_acc["region_class"] = diff_acc["region"].map(region_class_map).fillna("Unclassified")

# diff_acc["abs_effect"] = diff_acc["effect_size"].abs()
# top_candidates = (diff_acc.sort_values("abs_effect", ascending=False)
#                            .groupby("cell_type")
#                            .head(TOP_N_PER_CT)
#                            .sort_values(["cell_type", "abs_effect"], ascending=[True, False])
#                            .reset_index(drop=True))

# diff_acc.to_csv(os.path.join(FIG_OUT, "differential_accessibility_ALL_exploratory.csv"), index=False)
# top_candidates.to_csv(os.path.join(FIG_OUT, "differential_accessibility_TOP_candidates_exploratory.csv"), index=False)

# print(f"Full table: {diff_acc.shape[0]} region-tests (exploratory, no p-values reported)")
# print(f"Top {TOP_N_PER_CT} candidates per cell type saved: {top_candidates.shape[0]} rows total")
# print(top_candidates.groupby("cell_type")["region_class"].value_counts())

In [14]:
# In[A]: Full-data KO-vs-Ctrl accessibility effects with stratified bootstrap stability
#
# INTERPRETATION:
# - Point estimate = pseudobulk log2FC using ALL KO and ALL Ctrl metacells.
# - No metacells are discarded. Group-size imbalance is treated as a precision
#   difference, not corrected by dropping data.
# - Stability is assessed with a stratified bootstrap: each genotype group is
#   resampled WITH REPLACEMENT at its own true size (e.g., 38 KO, 8 Ctrl),
#   preserving actual sample sizes rather than forcing artificial balance.
# - No peak-level P values are reported. Metacells are not independent
#   biological replicates in this pooled-genotype design.

import os
import numpy as np
import pandas as pd

# =============================================================================
# Parameters
# =============================================================================

MIN_MEAN_ACC = 0.05
MIN_METACELLS_PER_GENOTYPE = 3

N_BOOTSTRAP = 1000
RANDOM_SEED = 42
PSEUDOCOUNT = 1e-3

# A region is called "stable" if its 95% bootstrap interval excludes zero
# AND its bootstrap sign consistency meets this threshold.
SIGN_CONSISTENCY_THRESHOLD = 0.80

# =============================================================================
# Analysis
# =============================================================================

rng = np.random.default_rng(RANDOM_SEED)
results = []
celltype_summary = []

cell_types_to_test = sorted(meta["cell_type"].dropna().unique())

for ct in cell_types_to_test:
    ct_mask = meta["cell_type"] == ct

    ko_cells = meta.index[
        ct_mask & (meta["genotype"] == "KO")
    ].intersection(acc_df.index)

    ctrl_cells = meta.index[
        ct_mask & (meta["genotype"] == "Ctrl")
    ].intersection(acc_df.index)

    n_ko = len(ko_cells)
    n_ctrl = len(ctrl_cells)

    if n_ko < MIN_METACELLS_PER_GENOTYPE or n_ctrl < MIN_METACELLS_PER_GENOTYPE:
        print(
            f"Skipping {ct}: KO={n_ko}, Ctrl={n_ctrl}; "
            f"requires >= {MIN_METACELLS_PER_GENOTYPE} metacells per genotype."
        )
        continue

    ko_mean_full = acc_df.loc[ko_cells].mean(axis=0)
    ctrl_mean_full = acc_df.loc[ctrl_cells].mean(axis=0)

    keep = (
        (ko_mean_full > MIN_MEAN_ACC) |
        (ctrl_mean_full > MIN_MEAN_ACC)
    )
    var_regions = acc_df.columns[keep]

    if len(var_regions) == 0:
        print(f"Skipping {ct}: no regions passed mean-accessibility filter.")
        continue

    ko_mat = acc_df.loc[ko_cells, var_regions].to_numpy(dtype=float)
    ctrl_mat = acc_df.loc[ctrl_cells, var_regions].to_numpy(dtype=float)
    n_regions = len(var_regions)

    # Full-data point estimate using ALL metacells. This is the primary effect.
    full_log2fc = (
        np.log2(ko_mat.mean(axis=0) + PSEUDOCOUNT) -
        np.log2(ctrl_mat.mean(axis=0) + PSEUDOCOUNT)
    )

    # Stratified bootstrap: resample each group WITH REPLACEMENT at its own
    # true size. No group is downsampled; no metacells are discarded.
    bootstrap_log2fc = np.empty((N_BOOTSTRAP, n_regions), dtype=np.float32)

    for b in range(N_BOOTSTRAP):
        ko_idx = rng.integers(0, n_ko, size=n_ko)
        ctrl_idx = rng.integers(0, n_ctrl, size=n_ctrl)

        ko_mean_b = ko_mat[ko_idx].mean(axis=0)
        ctrl_mean_b = ctrl_mat[ctrl_idx].mean(axis=0)

        bootstrap_log2fc[b] = (
            np.log2(ko_mean_b + PSEUDOCOUNT) -
            np.log2(ctrl_mean_b + PSEUDOCOUNT)
        )

    boot_q025 = np.quantile(bootstrap_log2fc, 0.025, axis=0)
    boot_q975 = np.quantile(bootstrap_log2fc, 0.975, axis=0)
    boot_median = np.median(bootstrap_log2fc, axis=0)

    positive_fraction = (bootstrap_log2fc > 0).mean(axis=0)
    negative_fraction = (bootstrap_log2fc < 0).mean(axis=0)
    sign_consistency = np.maximum(positive_fraction, negative_fraction)

    ci_excludes_zero = (boot_q025 > 0) | (boot_q975 < 0)
    stable_direction = (
        ci_excludes_zero &
        (sign_consistency >= SIGN_CONSISTENCY_THRESHOLD)
    )

    effect_direction = np.where(
        full_log2fc > 0, "KO_open",
        np.where(full_log2fc < 0, "KO_closed", "no_direction")
    )

    cell_result = pd.DataFrame({
        "region": var_regions,
        "cell_type": ct,
        "n_KO_metacells": n_ko,
        "n_Ctrl_metacells": n_ctrl,
        "n_bootstrap": N_BOOTSTRAP,
        "mean_accessibility_KO": ko_mean_full.loc[var_regions].to_numpy(),
        "mean_accessibility_Ctrl": ctrl_mean_full.loc[var_regions].to_numpy(),
        "full_log2FC": full_log2fc,
        "bootstrap_median_log2FC": boot_median,
        "bootstrap_q025_log2FC": boot_q025,
        "bootstrap_q975_log2FC": boot_q975,
        "sign_consistency": sign_consistency,
        "ci_excludes_zero": ci_excludes_zero,
        "stable_direction": stable_direction,
        "effect_direction": effect_direction,
        "abs_full_log2FC": np.abs(full_log2fc),
    })

    if "region_class_map" in globals():
        cell_result["region_class"] = (
            cell_result["region"].map(region_class_map).fillna("Unclassified")
        )
    else:
        cell_result["region_class"] = "Unclassified"

    results.append(cell_result)

    n_stable_open = int(
        ((cell_result["effect_direction"] == "KO_open") &
         cell_result["stable_direction"]).sum()
    )
    n_stable_closed = int(
        ((cell_result["effect_direction"] == "KO_closed") &
         cell_result["stable_direction"]).sum()
    )

    celltype_summary.append({
        "cell_type": ct,
        "n_KO_metacells": n_ko,
        "n_Ctrl_metacells": n_ctrl,
        "n_regions_tested": n_regions,
        "n_stable_KO_open": n_stable_open,
        "n_stable_KO_closed": n_stable_closed,
    })

    print(
        f"Completed {ct}: KO={n_ko}, Ctrl={n_ctrl}, regions={n_regions:,}, "
        f"stable KO-open={n_stable_open}, stable KO-closed={n_stable_closed}"
    )

if len(results) == 0:
    raise ValueError("No cell types met the minimum metacell requirement.")

diff_acc = pd.concat(results, ignore_index=True)
celltype_summary_df = pd.DataFrame(celltype_summary)

diff_acc_path = os.path.join(
    FIG_OUT, "exploratory_bootstrap_accessibility_effects_all_regions.csv"
)
summary_path = os.path.join(
    FIG_OUT, "exploratory_bootstrap_accessibility_celltype_summary.csv"
)

diff_acc.to_csv(diff_acc_path, index=False)
celltype_summary_df.to_csv(summary_path, index=False)

print(f"\nSaved full region-level table: {diff_acc_path}")
print(f"Saved per-cell-type summary: {summary_path}")
print("\n", celltype_summary_df.to_string(index=False))

Completed Astrocytes: KO=158, Ctrl=22, regions=352,799, stable KO-open=86924, stable KO-closed=123472
Completed Deep-layer extratelencephalic neurons: KO=60, Ctrl=14, regions=464,505, stable KO-open=141174, stable KO-closed=115091
Completed Endothelial cells: KO=22, Ctrl=4, regions=629,300, stable KO-open=300276, stable KO-closed=124171
Completed Layer 2/3 IT neurons: KO=18, Ctrl=4, regions=417,179, stable KO-open=88760, stable KO-closed=145230
Completed Layer 4 sensory neurons: KO=408, Ctrl=106, regions=450,370, stable KO-open=126173, stable KO-closed=245079
Completed Layer 5/6 IT neurons: KO=200, Ctrl=46, regions=461,648, stable KO-open=103525, stable KO-closed=170782
Completed Layer 5a IT neurons: KO=362, Ctrl=88, regions=449,264, stable KO-open=132893, stable KO-closed=216795
Completed Layer 5b PT neurons: KO=30, Ctrl=8, regions=469,651, stable KO-open=122000, stable KO-closed=179597
Completed Layer 6a corticothalamic neurons: KO=182, Ctrl=50, regions=445,154, stable KO-open=124413

In [27]:
# =============================================================================
# eRegulon-guided ATAC-RNA concordance test (gene-level, pre-specified TF targets)
# Reads everything fresh from disk.
# =============================================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.multitest import multipletests

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

BASE = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus"
FURTHER_OUT = os.path.join(BASE, "scenicplus_further_analysis")
FIG_OUT = os.path.join(FURTHER_OUT, "manuscript_figures")

CSV_DIR = os.path.join(FURTHER_OUT, "TF_target_csvs_multi_spec")
DEG_CSV = os.path.join(FURTHER_OUT, "DEG_KO_vs_Ctrl", "DEG_all_celltypes_KO_vs_Ctrl.csv")
DIFF_ACC_PATH = os.path.join(FIG_OUT, "exploratory_bootstrap_accessibility_effects_all_regions.csv")
GENES_PATH = os.path.join(FIG_OUT, "region_to_gene_nearest_FULL.csv")

OUT_DIR = os.path.join(FIG_OUT, "cross_modal_ATAC_RNA_concordance")
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
N_PERMUTATIONS = 10000
MIN_GENES_PER_TEST = 5
LOG2FC_COL = "bootstrap_median_log2FC"  # matches your actual diff_acc columns

rng = np.random.default_rng(RANDOM_SEED)

# Fixed cell-type color palette. NOTE: these are the LABELS used for plotting
# (with "+"), separate from the RAW names used in your TF-target CSV filenames
# (without "+"), which is what caused the matching bug.
celltype_order = [
    "Layer 2/3 IT neurons", "Layer 4 sensory neurons", "Layer 5a IT neurons",
    "Layer 5b PT neurons", "Layer 5/6 IT neurons", "Layer 6a corticothalamic neurons",
    "Layer 6b neurons", "Deep-layer extratelencephalic neurons",
    "Corticospinal neurons Type I", "Corticospinal neurons Type II",
    "PV+ interneurons", "SST+ interneurons", "VIP+ interneurons",
    "Astrocytes", "Oligodendrocyte precursor cells", "Oligodendrocytes",
    "Perineuronal oligodendrocytes", "Myelinating oligodendrocytes",
    "Microglia", "Endothelial cells", "Leptomeningeal cells", "Meningeal fibroblasts"
]
hex_colors = [
    "6B5B95", "45B8AC", "955251", "4E84C4", "B565A7", "88B04B", "7B6888",
    "C3447A", "009B77", "EFC050", "7FCDCD", "DD4124", "5B5EA6", "E07A5F",
    "4BACC6", "E8A0BF", "ea8a33", "9B2335", "C17BAE", "DECF3F", "789262", "BC243C"
]
atac_colors = {ct: f"#{h}" for ct, h in zip(celltype_order, hex_colors)}

# Raw names as they appear in your export loop / DEG table / diff_acc table
# (strip "+" before sanitizing, since your loop's ct_name source never had it)
raw_celltype_names = [ct.replace("+", "") for ct in celltype_order]
label_by_raw = {raw.strip(): label for raw, label in zip(raw_celltype_names, celltype_order)}


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9-]", "_", s)


# -----------------------------------------------------------------------------
# Step 1: rebuild csv_files, matching against RAW (unlabeled "+") cell-type names
# -----------------------------------------------------------------------------

safe_to_raw = {safe_name(raw): raw for raw in raw_celltype_names}

csv_files = []
unmatched = []
for path in glob.glob(os.path.join(CSV_DIR, "*.csv")):
    fname = os.path.basename(path)[:-4]
    matched = False
    for safe_ct, raw_ct in safe_to_raw.items():
        if fname.endswith(safe_ct):
            tf_name = fname[: -(len(safe_ct) + 1)]
            csv_files.append((tf_name, raw_ct, path))
            matched = True
            break
    if not matched:
        unmatched.append(fname)

print(f"Matched {len(csv_files)} TF-target CSVs to (TF, cell_type) pairs.")
if unmatched:
    print(f"Still unmatched ({len(unmatched)}): {unmatched}")


# -----------------------------------------------------------------------------
# Step 2: load DEG table
# -----------------------------------------------------------------------------

deg_all = pd.read_csv(DEG_CSV)
deg_all["gene"] = deg_all["gene"].astype(str)
print(f"DEG table columns: {deg_all.columns.tolist()}")
print(f"Total DEGs (significant): {deg_all['significant'].sum()}")


# -----------------------------------------------------------------------------
# Step 3: load bootstrap-stable ATAC effects, map region -> gene
# -----------------------------------------------------------------------------

diff_acc = pd.read_csv(DIFF_ACC_PATH)
print(f"Loaded diff_acc: {diff_acc.shape}, columns: {diff_acc.columns.tolist()}")

# Keep only bootstrap-stable regions, as intended
diff_acc = diff_acc[diff_acc["stable_direction"] == True].copy()
print(f"Bootstrap-stable regions: {diff_acc.shape}")

region_gene_map = None
if os.path.exists(GENES_PATH):
    region_gene_df = pd.read_csv(GENES_PATH)
    if "region" in region_gene_df.columns and "gene" in region_gene_df.columns:
        region_gene_map = (
            region_gene_df.dropna(subset=["gene"])
            .drop_duplicates(subset="region")
            .set_index("region")["gene"]
        )
        print(f"Loaded region->gene map: {len(region_gene_map)} regions "
              f"(NOTE: this is only the TOP-candidates subset per cell type, "
              f"not the full stable-region universe)")

if region_gene_map is not None:
    diff_acc["gene"] = diff_acc["region"].map(region_gene_map)
else:
    diff_acc["gene"] = np.nan

diff_acc = diff_acc.dropna(subset=["gene"]).copy()
diff_acc["gene"] = diff_acc["gene"].astype(str)
diff_acc["effect_direction"] = np.where(diff_acc[LOG2FC_COL] > 0, "KO_open", "KO_closed")

gene_atac = (
    diff_acc.groupby(["cell_type", "gene"], as_index=False)
    .agg(mean_log2FC=(LOG2FC_COL, "mean"))
)
gene_atac["effect_direction"] = np.where(gene_atac["mean_log2FC"] > 0, "KO_open", "KO_closed")
gene_atac["stable_direction"] = True

print(f"gene_atac (region->gene collapsed, per cell type): {gene_atac.shape}")
if gene_atac.shape[0] < 50:
    print("WARNING: very few region->gene mappings available (top-candidates export "
          "only covers ~30 genes per cell type). Re-run the region_to_gene_nearest merge "
          "against the FULL stable-region table (not just top candidates) for full power.")


# -----------------------------------------------------------------------------
# Step 4: core gene-level concordance test, restricted to each TF's own
# rho>0 regulon targets
# -----------------------------------------------------------------------------

def run_eregulon_guided_concordance(csv_files, gene_atac, deg_all,
                                     n_perms=N_PERMUTATIONS,
                                     min_genes=MIN_GENES_PER_TEST,
                                     seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    results = []
    gene_level_rows = []

    deg_col_ct = "celltype" if "celltype" in deg_all.columns else "cell_type"

    for tf_name, ct_name, csv_path in csv_files:
        tf_target_df = pd.read_csv(csv_path)
        if "rho" not in tf_target_df.columns or "target" not in tf_target_df.columns:
            continue
        tf_targets = set(
            tf_target_df[tf_target_df["rho"] > 0]["target"]
            .dropna().astype(str).str.strip()
        )
        if not tf_targets:
            continue

        atac_sub = gene_atac[
            (gene_atac["cell_type"] == ct_name) & (gene_atac["gene"].isin(tf_targets))
        ][["gene", "effect_direction"]]

        deg_sub = deg_all[
            (deg_all[deg_col_ct] == ct_name) & (deg_all["gene"].isin(tf_targets))
        ][["gene", "log2FC", "significant"]]
        deg_sub = deg_sub[deg_sub["significant"] == True]

        merged = atac_sub.merge(deg_sub[["gene", "log2FC"]], on="gene", how="inner")
        if merged.shape[0] < min_genes:
            continue

        merged["rna_direction"] = np.where(merged["log2FC"] > 0, "up", "down")

        a = int(((merged["effect_direction"] == "KO_open") & (merged["rna_direction"] == "up")).sum())
        b = int(((merged["effect_direction"] == "KO_open") & (merged["rna_direction"] == "down")).sum())
        c = int(((merged["effect_direction"] == "KO_closed") & (merged["rna_direction"] == "up")).sum())
        d = int(((merged["effect_direction"] == "KO_closed") & (merged["rna_direction"] == "down")).sum())

        table = np.array([[a, b], [c, d]])
        n_concordant = a + d
        n_discordant = b + c
        n_total = n_concordant + n_discordant

        _, fisher_p = fisher_exact(table, alternative="two-sided")
        ci_table = Table2x2(table.astype(float) + 0.5)
        odds_ratio = ci_table.oddsratio
        ci_low, ci_high = ci_table.oddsratio_confint(alpha=0.05)

        atac_is_open = (merged["effect_direction"] == "KO_open").to_numpy()
        rna_dir_arr = merged["rna_direction"].to_numpy()
        n_up_observed = int((rna_dir_arr == "up").sum())
        n_genes = len(merged)

        perm_concordant = np.empty(n_perms, dtype=int)
        idx = np.arange(n_genes)
        for i in range(n_perms):
            perm_up_idx = rng_local.choice(idx, size=n_up_observed, replace=False)
            perm_is_up = np.zeros(n_genes, dtype=bool)
            perm_is_up[perm_up_idx] = True
            perm_concordant[i] = int(
                ((atac_is_open & perm_is_up) | (~atac_is_open & ~perm_is_up)).sum()
            )

        empirical_p = (np.sum(perm_concordant >= n_concordant) + 1) / (n_perms + 1)
        expected_concordant = perm_concordant.mean()

        results.append({
            "TF": tf_name, "cell_type": label_by_raw.get(ct_name, ct_name),
            "n_regulon_genes_tested": n_total,
            "n_KO_open_RNA_up": a, "n_KO_open_RNA_down": b,
            "n_KO_closed_RNA_up": c, "n_KO_closed_RNA_down": d,
            "n_concordant": n_concordant, "n_discordant": n_discordant,
            "concordant_fraction": n_concordant / n_total,
            "odds_ratio_descriptive": odds_ratio,
            "odds_ratio_95CI_low": ci_low, "odds_ratio_95CI_high": ci_high,
            "fisher_p_two_sided": fisher_p,
            "empirical_permutation_p": empirical_p,
            "expected_concordant_permutation": expected_concordant,
        })

        merged["TF"] = tf_name
        merged["cell_type"] = label_by_raw.get(ct_name, ct_name)
        gene_level_rows.append(merged)

    results_df = pd.DataFrame(results)
    if results_df.empty:
        raise ValueError(
            "No TF/cell-type pairs had enough overlap to test. "
            "Most likely cause: the region->gene mapping only covers top-candidate "
            "genes (~30/cell type), too sparse to overlap with most TFs' targets. "
            "Re-export a full stable-region -> nearest-gene mapping."
        )

    results_df["permutation_FDR_BH"] = multipletests(
        results_df["empirical_permutation_p"].fillna(1.0), method="fdr_bh"
    )[1]
    results_df["fisher_FDR_BH"] = multipletests(
        results_df["fisher_p_two_sided"].fillna(1.0), method="fdr_bh"
    )[1]
    results_df = results_df.sort_values("permutation_FDR_BH").reset_index(drop=True)

    gene_level_df = pd.concat(gene_level_rows, ignore_index=True) if gene_level_rows else pd.DataFrame()
    return results_df, gene_level_df


# -----------------------------------------------------------------------------
# Step 5: plotting function
# -----------------------------------------------------------------------------

def plot_eregulon_guided_concordance(results_df, out_dir=OUT_DIR, top_n_examples=4):
    df = results_df.copy()
    df["color"] = df["cell_type"].map(atac_colors).fillna("#999999")
    df["significant"] = df["permutation_FDR_BH"] < 0.05
    df["log2_OR"] = np.log2(df["odds_ratio_descriptive"].clip(lower=0.02))
    df["log2_OR_CI_low"] = np.log2(df["odds_ratio_95CI_low"].clip(lower=0.02))
    df["log2_OR_CI_high"] = np.log2(df["odds_ratio_95CI_high"].clip(lower=0.02))
    df["label"] = df["TF"] + " (" + df["cell_type"] + ")"
    df = df.sort_values("log2_OR", ascending=True).reset_index(drop=True)
    df["y_pos"] = np.arange(len(df))

    fig_w_in = 24 / 2.54
    fig_h_in = max(10, 0.28 * len(df))
    fig = plt.figure(figsize=(fig_w_in, fig_h_in))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.3)

    ax_or = fig.add_subplot(gs[0])
    ax_or.errorbar(
        df["log2_OR"], df["y_pos"],
        xerr=[df["log2_OR"] - df["log2_OR_CI_low"], df["log2_OR_CI_high"] - df["log2_OR"]],
        fmt="none", ecolor="#888888", elinewidth=1.2, capsize=2.5, zorder=1
    )
    ax_or.scatter(df["log2_OR"], df["y_pos"], s=90, c=df["color"],
                  edgecolor="black", linewidth=0.6, zorder=3)
    ax_or.axvline(0, color="black", linestyle="--", linewidth=1.1, zorder=0)
    for _, row in df[df["significant"]].iterrows():
        ax_or.text(row["log2_OR"] + 0.15, row["y_pos"], "*",
                    fontsize=13, fontweight="bold", color="black", va="center")
    ax_or.set_yticks(df["y_pos"])
    ax_or.set_yticklabels(df["label"], fontsize=7)
    for tick, ct in zip(ax_or.get_yticklabels(), df["cell_type"]):
        tick.set_color(atac_colors.get(ct, "#333333"))
    ax_or.set_xlabel("log2(Odds ratio): ATAC direction vs RNA direction\n"
                      "(within each TF's own regulon targets, rho>0)", fontsize=10)
    ax_or.set_title(
        "eRegulon-guided ATAC-RNA concordance\n"
        "Gene-level test, pre-specified TF regulon targets only, * FDR<0.05",
        fontsize=12, fontweight="bold"
    )
    ax_or.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.5)
    ax_or.spines[["top", "right"]].set_visible(False)

    top_pairs = df[df["significant"]].sort_values("permutation_FDR_BH").head(top_n_examples)
    if top_pairs.empty:
        top_pairs = df.reindex(df["log2_OR"].abs().sort_values(ascending=False).index).head(top_n_examples)

    gs_bottom = gs[1].subgridspec(1, max(len(top_pairs), 1), wspace=0.5)
    for i, (_, row) in enumerate(top_pairs.iterrows()):
        ax = fig.add_subplot(gs_bottom[0, i])
        color = row["color"]
        counts = [row["n_concordant"], row["n_discordant"]]
        bars = ax.bar(["Concordant", "Discordant"], counts,
                       color=[color, "#dddddd"], edgecolor="black", linewidth=0.6)
        for bar, val in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3, str(val), ha="center", fontsize=8)
        ax.set_title(f"{row['TF']}\n{row['cell_type']}", fontsize=8.5, color=color)
        ax.set_ylabel("N genes", fontsize=8)
        ax.tick_params(labelsize=7.5)
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    out_pdf = os.path.join(out_dir, "Fig_eRegulon_guided_ATAC_RNA_concordance.pdf")
    out_png = out_pdf.replace(".pdf", ".png")
    fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
    fig.savefig(out_png, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {out_pdf}")
    print(f"Saved: {out_png}")


# -----------------------------------------------------------------------------
# Execute
# -----------------------------------------------------------------------------

results_df, gene_level_df = run_eregulon_guided_concordance(csv_files, gene_atac, deg_all)

results_path = os.path.join(OUT_DIR, "eRegulon_guided_ATAC_RNA_concordance_geneLevel.csv")
gene_path = os.path.join(OUT_DIR, "eRegulon_guided_ATAC_RNA_geneLevel_details.csv")
results_df.to_csv(results_path, index=False)
gene_level_df.to_csv(gene_path, index=False)
print(f"Saved: {results_path}")
print(f"Saved: {gene_path}")

print("\n=== eRegulon-guided gene-level concordance results ===")
print(results_df[[
    "TF", "cell_type", "n_regulon_genes_tested", "n_concordant", "n_discordant",
    "odds_ratio_descriptive", "empirical_permutation_p", "permutation_FDR_BH"
]].to_string(index=False))

plot_eregulon_guided_concordance(results_df)

Matched 85 TF-target CSVs to (TF, cell_type) pairs.
DEG table columns: ['gene', 'log2FC', 'pval', 'padj', 'cell_type', 'significant']
Total DEGs (significant): 13201
Loaded diff_acc: (7266857, 17), columns: ['region', 'cell_type', 'n_KO_metacells', 'n_Ctrl_metacells', 'n_bootstrap', 'mean_accessibility_KO', 'mean_accessibility_Ctrl', 'full_log2FC', 'bootstrap_median_log2FC', 'bootstrap_q025_log2FC', 'bootstrap_q975_log2FC', 'sign_consistency', 'ci_excludes_zero', 'stable_direction', 'effect_direction', 'abs_full_log2FC', 'region_class']
Bootstrap-stable regions: (4774175, 17)
Loaded region->gene map: 727348 regions (NOTE: this is only the TOP-candidates subset per cell type, not the full stable-region universe)
gene_atac (region->gene collapsed, per cell type): (266157, 5)
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/eRegulon_guided_ATAC_RNA_concordance_geneLevel.csv
Saved: /U

/var/folders/c5/9gnt7c7x7x59ygrp90w3n60r0000gn/T/ipykernel_38693/1563841928.py:328: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.98])


Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_guided_ATAC_RNA_concordance.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_guided_ATAC_RNA_concordance.png


In [64]:
# =============================================================================
# eRegulon-guided ATAC-RNA concordance -- POOLED version (recovers power)
# Primary test: pool all TFs' rho>0 targets per (cell_type, direction)
# Secondary: per-TF breakdown, clearly labeled exploratory / underpowered
# =============================================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.multitest import multipletests

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

BASE = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus"
FURTHER_OUT = os.path.join(BASE, "scenicplus_further_analysis")
FIG_OUT = os.path.join(FURTHER_OUT, "manuscript_figures")

CSV_DIR = os.path.join(FURTHER_OUT, "TF_target_csvs_multi_spec")
DEG_CSV = os.path.join(FURTHER_OUT, "DEG_KO_vs_Ctrl", "DEG_all_celltypes_KO_vs_Ctrl.csv")
DIFF_ACC_PATH = os.path.join(FIG_OUT, "exploratory_bootstrap_accessibility_effects_all_regions.csv")
GENES_PATH = os.path.join(FIG_OUT, "region_to_gene_nearest_FULL.csv")

OUT_DIR = os.path.join(FIG_OUT, "cross_modal_ATAC_RNA_concordance")
os.makedirs(OUT_DIR, exist_ok=True)

RESULTS_POOLED_CSV = os.path.join(OUT_DIR, "eRegulon_pooled_ATAC_RNA_concordance_v2.csv")
RESULTS_PERTF_CSV = os.path.join(OUT_DIR, "eRegulon_perTF_ATAC_RNA_concordance_exploratory_v2.csv")
GENE_LEVEL_CSV = os.path.join(OUT_DIR, "eRegulon_pooled_ATAC_RNA_geneLevel_details_v2.csv")
FIG_PDF = os.path.join(OUT_DIR, "Fig_eRegulon_pooled_ATAC_RNA_concordance_v2.pdf")
FIG_PNG = FIG_PDF.replace(".pdf", ".png")

RANDOM_SEED = 42
N_PERMUTATIONS = 10000
MIN_GENES_PER_TEST_POOLED = 15
MIN_GENES_PER_TEST_PERTF = 5
LOG2FC_COL = "bootstrap_median_log2FC"

rng = np.random.default_rng(RANDOM_SEED)

celltype_order = [
    'Layer 2/3 IT neurons', 'Layer 4 sensory neurons', 'Layer 5a IT neurons',
    'Layer 5b PT neurons', 'Layer 5/6 IT neurons', 'Layer 6a corticothalamic neurons',
    'Layer 6b neurons', 'Deep-layer extratelencephalic neurons',
    'Corticospinal neurons Type I', 'Corticospinal neurons Type II',
    'PV+ interneurons', 'SST+ interneurons', 'VIP+ interneurons',
    'Astrocytes', 'Oligodendrocyte precursor cells', 'Oligodendrocytes',
    'Microglia', 'Endothelial cells', 'Leptomeningeal cells', 'Meningeal fibroblasts'
]

hex_colors = [
    '6B5B95', '45B8AC', '955251', '4E84C4', 'B565A7', '88B04B', '7B6888', 'C3447A',
    '009B77', 'EFC050', '7FCDCD', 'DD4124', '5B5EA6', 'E07A5F', '4BACC6', 'E8A0BF',
    'C17BAE', 'DECF3F', '789262', 'BC243C'
]

atac_colors = {ct: f"#{h}" for ct, h in zip(celltype_order, hex_colors)}

# raw_ct (no "+") is used ONLY for filename matching against the TF-target CSVs.
# label_by_raw maps back to the canonical label (WITH "+") that gene_atac and
# deg_all actually use in their cell_type / celltype columns.
raw_celltype_names = [ct.replace("+", "") for ct in celltype_order]
label_by_raw = {raw.strip(): label for raw, label in zip(raw_celltype_names, celltype_order)}


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9-]", "_", s)


# -----------------------------------------------------------------------------
# Step 1: match TF-target CSVs to (TF, cell_type)
# -----------------------------------------------------------------------------

safe_to_raw = {safe_name(raw): raw for raw in raw_celltype_names}

csv_files = []
unmatched = []
for path in glob.glob(os.path.join(CSV_DIR, "*.csv")):
    fname = os.path.basename(path)[:-4]
    matched = False
    for safe_ct, raw_ct in safe_to_raw.items():
        if fname.endswith(safe_ct):
            tf_name = fname[: -(len(safe_ct) + 1)]
            csv_files.append((tf_name, raw_ct, path))
            matched = True
            break
    if not matched:
        unmatched.append(fname)

print(f"Matched {len(csv_files)} TF-target CSVs to (TF, cell_type) pairs.")
if unmatched:
    print(f"Still unmatched ({len(unmatched)}): {unmatched}")


# -----------------------------------------------------------------------------
# Step 2: load DEG table
# -----------------------------------------------------------------------------

deg_all = pd.read_csv(DEG_CSV)
deg_all["gene"] = deg_all["gene"].astype(str)
deg_col_ct = "celltype" if "celltype" in deg_all.columns else "cell_type"
print(f"Total DEGs (significant): {deg_all['significant'].sum()}")

# Sanity check: confirm which cell types actually carry "+" in the real data,
# so canonical-label filtering is guaranteed to match downstream.
print("Cell types in deg_all:", sorted(deg_all[deg_col_ct].unique()))


# -----------------------------------------------------------------------------
# Step 3: load bootstrap-stable ATAC effects, map region -> gene
# -----------------------------------------------------------------------------

diff_acc = pd.read_csv(DIFF_ACC_PATH)
diff_acc = diff_acc[diff_acc["stable_direction"] == True].copy()
print(f"Bootstrap-stable regions: {diff_acc.shape}")

region_gene_df = pd.read_csv(GENES_PATH)
region_gene_map = (
    region_gene_df.dropna(subset=["gene"])
    .drop_duplicates(subset="region")
    .set_index("region")["gene"]
)
print(f"Loaded FULL region->gene map: {len(region_gene_map)} regions")

diff_acc["gene"] = diff_acc["region"].map(region_gene_map)
diff_acc = diff_acc.dropna(subset=["gene"]).copy()
diff_acc["gene"] = diff_acc["gene"].astype(str)
diff_acc["effect_direction"] = np.where(diff_acc[LOG2FC_COL] > 0, "KO_open", "KO_closed")

gene_atac = (
    diff_acc.groupby(["cell_type", "gene"], as_index=False)
    .agg(mean_log2FC=(LOG2FC_COL, "mean"))
)
gene_atac["effect_direction"] = np.where(gene_atac["mean_log2FC"] > 0, "KO_open", "KO_closed")
print(f"gene_atac (region->gene collapsed, per cell type): {gene_atac.shape}")
print("Cell types in gene_atac:", sorted(gene_atac["cell_type"].unique()))


# -----------------------------------------------------------------------------
# Step 4: build pooled rho>0 target sets per (TF, cell_type), then union
# across all TFs sharing a cell type
# -----------------------------------------------------------------------------

tf_targets_by_ct = {}  # raw_ct (no "+") -> set of genes, union across all TFs
for tf_name, ct_name, csv_path in csv_files:
    tf_df = pd.read_csv(csv_path)
    if "rho" not in tf_df.columns or "target" not in tf_df.columns:
        continue
    targets = set(tf_df[tf_df["rho"] > 0]["target"].dropna().astype(str).str.strip())
    if not targets:
        continue
    tf_targets_by_ct.setdefault(ct_name, set()).update(targets)

print(f"Cell types with pooled TF-target gene sets: {len(tf_targets_by_ct)}")
for ct, genes in tf_targets_by_ct.items():
    print(f"  {label_by_raw.get(ct, ct)}: {len(genes)} pooled regulon-target genes")


# -----------------------------------------------------------------------------
# Step 5: PRIMARY pooled test, per (cell_type, direction)
# FIX: filter gene_atac / deg_all using the CANONICAL label (with "+"),
# not the stripped raw_ct used for filename matching.
# -----------------------------------------------------------------------------

def run_pooled_concordance(tf_targets_by_ct, gene_atac, deg_all, deg_col_ct, label_by_raw,
                            n_perms=N_PERMUTATIONS, min_genes=MIN_GENES_PER_TEST_POOLED,
                            seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    rows = []

    for ct_name, pooled_targets in tf_targets_by_ct.items():
        canonical_ct = label_by_raw.get(ct_name, ct_name)  # e.g. "PV+ interneurons"

        atac_sub = gene_atac[
            (gene_atac["cell_type"] == canonical_ct) & (gene_atac["gene"].isin(pooled_targets))
        ][["gene", "effect_direction"]]

        deg_sub = deg_all[
            (deg_all[deg_col_ct] == canonical_ct) & (deg_all["gene"].isin(pooled_targets)) & (deg_all["significant"] == True)
        ][["gene", "log2FC"]]

        merged = atac_sub.merge(deg_sub, on="gene", how="inner")
        if merged.shape[0] < min_genes:
            print(f"Skipping {canonical_ct}: only {merged.shape[0]} genes overlap (min {min_genes})")
            continue

        merged["rna_direction"] = np.where(merged["log2FC"] > 0, "up", "down")

        for direction_label, atac_dir, rna_dir in [
            ("KO-open / RNA-up", "KO_open", "up"),
            ("KO-closed / RNA-down", "KO_closed", "down"),
        ]:
            n_selected = merged.shape[0]
            n_concordant = int(
                ((merged["effect_direction"] == atac_dir) & (merged["rna_direction"] == rna_dir)).sum()
            )
            other_atac_dir = "KO_closed" if atac_dir == "KO_open" else "KO_open"
            other_rna_dir = "down" if rna_dir == "up" else "up"

            a = n_concordant
            b = int(((merged["effect_direction"] == atac_dir) & (merged["rna_direction"] == other_rna_dir)).sum())
            c = int(((merged["effect_direction"] == other_atac_dir) & (merged["rna_direction"] == rna_dir)).sum())
            d = int(((merged["effect_direction"] == other_atac_dir) & (merged["rna_direction"] == other_rna_dir)).sum())

            table = np.array([[a, b], [c, d]])
            _, fisher_p = fisher_exact(table, alternative="two-sided")
            ci_table = Table2x2(table.astype(float) + 0.5)
            odds_ratio = ci_table.oddsratio
            ci_low, ci_high = ci_table.oddsratio_confint(alpha=0.05)

            atac_is_target_dir = (merged["effect_direction"] == atac_dir).to_numpy()
            rna_is_target_dir = (merged["rna_direction"] == rna_dir).to_numpy()
            n_rna_hits_observed = int(rna_is_target_dir.sum())
            n_genes = n_selected
            idx = np.arange(n_genes)

            perm_hits = np.empty(n_perms, dtype=int)
            for i in range(n_perms):
                perm_up_idx = rng_local.choice(idx, size=n_rna_hits_observed, replace=False)
                perm_mask = np.zeros(n_genes, dtype=bool)
                perm_mask[perm_up_idx] = True
                perm_hits[i] = int((atac_is_target_dir & perm_mask).sum())

            empirical_p = (np.sum(perm_hits >= a) + 1) / (n_perms + 1)

            rows.append({
                "cell_type": canonical_ct,
                "direction_label": direction_label,
                "n_pooled_regulon_genes_tested": n_selected,
                "n_concordant": a,
                "n_discordant_atacdir_rna_other": b,
                "n_discordant_other_atacdir_rnadir": c,
                "n_neither": d,
                "odds_ratio_descriptive": odds_ratio,
                "odds_ratio_95CI_low": ci_low,
                "odds_ratio_95CI_high": ci_high,
                "fisher_p_descriptive": fisher_p,
                "empirical_permutation_p": empirical_p,
                "expected_concordant_permutation": perm_hits.mean(),
            })

    results_df = pd.DataFrame(rows)
    if results_df.empty:
        raise ValueError("No pooled cell-type tests met the minimum gene threshold.")

    results_df["permutation_FDR_BH"] = multipletests(
        results_df["empirical_permutation_p"].fillna(1.0), method="fdr_bh"
    )[1]
    results_df = results_df.sort_values("permutation_FDR_BH").reset_index(drop=True)
    return results_df


pooled_results = run_pooled_concordance(tf_targets_by_ct, gene_atac, deg_all, deg_col_ct, label_by_raw)
pooled_results.to_csv(RESULTS_POOLED_CSV, index=False)
print(f"Saved: {RESULTS_POOLED_CSV}")
print("\n=== PRIMARY pooled concordance results (per cell type, per direction) ===")
print(pooled_results[[
    "cell_type", "direction_label", "n_pooled_regulon_genes_tested", "n_concordant",
    "odds_ratio_descriptive", "empirical_permutation_p", "permutation_FDR_BH"
]].to_string(index=False))


# -----------------------------------------------------------------------------
# Step 6: SECONDARY per-TF exploratory test
# FIX: same canonical-label filtering applied here as well.
# -----------------------------------------------------------------------------

def run_pertf_exploratory(csv_files, gene_atac, deg_all, deg_col_ct, label_by_raw,
                           n_perms=N_PERMUTATIONS, min_genes=MIN_GENES_PER_TEST_PERTF,
                           seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    results = []
    gene_level_rows = []

    for tf_name, ct_name, csv_path in csv_files:
        canonical_ct = label_by_raw.get(ct_name, ct_name)

        tf_target_df = pd.read_csv(csv_path)
        if "rho" not in tf_target_df.columns or "target" not in tf_target_df.columns:
            continue
        tf_targets = set(tf_target_df[tf_target_df["rho"] > 0]["target"].dropna().astype(str).str.strip())
        if not tf_targets:
            continue

        atac_sub = gene_atac[
            (gene_atac["cell_type"] == canonical_ct) & (gene_atac["gene"].isin(tf_targets))
        ][["gene", "effect_direction"]]
        deg_sub = deg_all[
            (deg_all[deg_col_ct] == canonical_ct) & (deg_all["gene"].isin(tf_targets)) & (deg_all["significant"] == True)
        ][["gene", "log2FC"]]

        merged = atac_sub.merge(deg_sub, on="gene", how="inner")
        if merged.shape[0] < min_genes:
            continue

        merged["rna_direction"] = np.where(merged["log2FC"] > 0, "up", "down")
        a = int(((merged["effect_direction"] == "KO_open") & (merged["rna_direction"] == "up")).sum())
        b = int(((merged["effect_direction"] == "KO_open") & (merged["rna_direction"] == "down")).sum())
        c = int(((merged["effect_direction"] == "KO_closed") & (merged["rna_direction"] == "up")).sum())
        d = int(((merged["effect_direction"] == "KO_closed") & (merged["rna_direction"] == "down")).sum())

        table = np.array([[a, b], [c, d]])
        n_concordant = a + d
        _, fisher_p = fisher_exact(table, alternative="two-sided")
        ci_table = Table2x2(table.astype(float) + 0.5)
        odds_ratio = ci_table.oddsratio
        ci_low, ci_high = ci_table.oddsratio_confint(alpha=0.05)

        atac_is_open = (merged["effect_direction"] == "KO_open").to_numpy()
        rna_dir_arr = merged["rna_direction"].to_numpy()
        n_up_observed = int((rna_dir_arr == "up").sum())
        n_genes = len(merged)
        idx = np.arange(n_genes)

        perm_concordant = np.empty(n_perms, dtype=int)
        for i in range(n_perms):
            perm_up_idx = rng_local.choice(idx, size=n_up_observed, replace=False)
            perm_is_up = np.zeros(n_genes, dtype=bool)
            perm_is_up[perm_up_idx] = True
            perm_concordant[i] = int(((atac_is_open & perm_is_up) | (~atac_is_open & ~perm_is_up)).sum())

        empirical_p = (np.sum(perm_concordant >= n_concordant) + 1) / (n_perms + 1)

        results.append({
            "TF": tf_name, "cell_type": canonical_ct,
            "n_regulon_genes_tested": n_concordant + b + c,
            "n_concordant": n_concordant,
            "odds_ratio_descriptive": odds_ratio,
            "odds_ratio_95CI_low": ci_low, "odds_ratio_95CI_high": ci_high,
            "fisher_p_two_sided": fisher_p,
            "empirical_permutation_p": empirical_p,
        })
        merged["TF"] = tf_name
        merged["cell_type"] = canonical_ct
        gene_level_rows.append(merged)

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df["permutation_FDR_BH"] = multipletests(
            results_df["empirical_permutation_p"].fillna(1.0), method="fdr_bh"
        )[1]
        results_df = results_df.sort_values("permutation_FDR_BH").reset_index(drop=True)
    gene_level_df = pd.concat(gene_level_rows, ignore_index=True) if gene_level_rows else pd.DataFrame()
    return results_df, gene_level_df


pertf_results, gene_level_df = run_pertf_exploratory(csv_files, gene_atac, deg_all, deg_col_ct, label_by_raw)
pertf_results.to_csv(RESULTS_PERTF_CSV, index=False)
gene_level_df.to_csv(GENE_LEVEL_CSV, index=False)
print(f"Saved: {RESULTS_PERTF_CSV}")
print(f"Saved: {GENE_LEVEL_CSV}")

Matched 85 TF-target CSVs to (TF, cell_type) pairs.
Total DEGs (significant): 13201
Cell types in deg_all: ['Astrocytes', 'Corticospinal neurons (Type I)', 'Deep-layer extratelencephalic neurons', 'Endothelial cells', 'Layer 2/3 IT neurons', 'Layer 4 sensory neurons', 'Layer 5/6 IT neurons', 'Layer 5a IT neurons', 'Layer 5b PT neurons', 'Layer 6a corticothalamic neurons', 'Layer 6b neurons', 'Leptomeningeal cells', 'Meningeal fibroblasts', 'Microglia', 'Oligodendrocyte precursor cells', 'Oligodendrocytes', 'PV+ interneurons', 'SST+ interneurons', 'VIP+ interneurons']
Bootstrap-stable regions: (4774175, 17)
Loaded FULL region->gene map: 727348 regions
gene_atac (region->gene collapsed, per cell type): (266157, 4)
Cell types in gene_atac: ['Astrocytes', 'Deep-layer extratelencephalic neurons', 'Endothelial cells', 'Layer 2/3 IT neurons', 'Layer 4 sensory neurons', 'Layer 5/6 IT neurons', 'Layer 5a IT neurons', 'Layer 5b PT neurons', 'Layer 6a corticothalamic neurons', 'Meningeal fibrobla

In [58]:
# # -----------------------------------------------------------------------------
# # Step 7: Plot -- primary pooled forest plot (top) + per-TF exploratory
# # dot strip (bottom), both on one figure
# # -----------------------------------------------------------------------------

# def plot_pooled_and_exploratory(pooled_df, pertf_df, out_pdf=FIG_PDF, out_png=FIG_PNG):
#     df = pooled_df.copy()
#     df["color"] = df["cell_type"].map(atac_colors).fillna("#999999")
#     df["significant"] = df["permutation_FDR_BH"] < 0.05
#     df["log2_OR"] = np.log2(df["odds_ratio_descriptive"].clip(lower=0.02))
#     df["log2_OR_CI_low"] = np.log2(df["odds_ratio_95CI_low"].clip(lower=0.02))
#     df["log2_OR_CI_high"] = np.log2(df["odds_ratio_95CI_high"].clip(lower=0.02))
#     df["label"] = df["cell_type"] + " (" + df["direction_label"] + ")"
#     df = df.sort_values("log2_OR", ascending=True).reset_index(drop=True)
#     df["y_pos"] = np.arange(len(df))

#     # Exact 300:760 (width:height) aspect ratio, scaled up for print resolution
#     ASPECT_W, ASPECT_H = 300, 760
#     fig_w_in = 8.0
#     fig_h_in = fig_w_in * (ASPECT_H / ASPECT_W)

#     fig = plt.figure(figsize=(fig_w_in, fig_h_in))
#     gs = fig.add_gridspec(2, 1, height_ratios=[2.6, 1], hspace=0.32)

#     ax_top = fig.add_subplot(gs[0])
#     # Thinner, semi-transparent whiskers -- de-emphasized relative to the dots
#     ax_top.errorbar(
#         df["log2_OR"], df["y_pos"],
#         xerr=[df["log2_OR"] - df["log2_OR_CI_low"], df["log2_OR_CI_high"] - df["log2_OR"]],
#         fmt="none", ecolor="#aaaaaa", elinewidth=0.7, capsize=2, capthick=0.7,
#         alpha=0.6, zorder=1
#     )
#     dot_size = np.where(df["significant"], 130, 75)
#     ax_top.scatter(df["log2_OR"], df["y_pos"], s=dot_size, c=df["color"],
#                    edgecolor="black", linewidth=0.7, zorder=3)
#     ax_top.axvline(0, color="black", linestyle="--", linewidth=1.0, zorder=0)
#     for _, row in df[df["significant"]].iterrows():
#         ax_top.text(row["log2_OR"] + 0.15, row["y_pos"], "*",
#                     fontsize=13, fontweight="bold", color="black", va="center")
#     ax_top.set_yticks(df["y_pos"])
#     # fontweight="bold" keeps the hex colors at full saturation instead of washing out
#     ax_top.set_yticklabels(df["label"], fontsize=7.5, fontweight="bold")
#     for tick, ct in zip(ax_top.get_yticklabels(), df["cell_type"]):
#         tick.set_color(atac_colors.get(ct, "#333333"))
#     ax_top.set_xlabel("log2(Odds ratio): ATAC direction vs RNA direction", fontsize=9.5)
#     ax_top.set_title(
#         "PRIMARY: pooled eRegulon-guided ATAC-RNA concordance\n"
#         "All TFs' rho>0 targets pooled per cell type and direction, * FDR<0.05",
#         fontsize=10.5, fontweight="bold"
#     )
#     ax_top.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.4)
#     ax_top.spines[["top", "right"]].set_visible(False)

#     ax_bottom = fig.add_subplot(gs[1])
#     pertf_plot = pertf_df.copy()
#     pertf_plot["color"] = pertf_plot["cell_type"].map(atac_colors).fillna("#999999")
#     pertf_plot["log2_OR"] = np.log2(pertf_plot["odds_ratio_descriptive"].clip(lower=0.02))
#     pertf_plot["significant"] = pertf_plot["permutation_FDR_BH"] < 0.05
#     x_positions = np.arange(len(pertf_plot))
#     ax_bottom.scatter(
#         x_positions, pertf_plot["log2_OR"],
#         s=pertf_plot["n_regulon_genes_tested"].clip(upper=40) * 1.6 + 8,
#         c=pertf_plot["color"], edgecolor="black", linewidth=0.4,
#         alpha=np.where(pertf_plot["significant"], 0.95, 0.4)
#     )
#     ax_bottom.axhline(0, color="black", linestyle="--", linewidth=0.9)
#     ax_bottom.set_xticks([])
#     ax_bottom.set_ylabel("log2(OR), per-TF", fontsize=8.5)
#     ax_bottom.set_xlabel(
#         f"Individual TF/cell-type pairs (n={len(pertf_plot)}), sorted arbitrarily\n"
#         "EXPLORATORY ONLY -- most tests underpowered (n genes typically <20), no FDR-significant hits expected",
#         fontsize=7.5, style="italic"
#     )
#     ax_bottom.set_title("SECONDARY: per-TF breakdown (descriptive, not independently powered)",
#                          fontsize=9.5, fontweight="bold", color="#555555")
#     ax_bottom.grid(axis="y", linestyle=":", linewidth=0.4, alpha=0.4)
#     ax_bottom.spines[["top", "right"]].set_visible(False)

#     plt.tight_layout()
#     fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
#     fig.savefig(out_png, bbox_inches="tight", dpi=300)
#     plt.close(fig)
#     print(f"Saved: {out_pdf}")
#     print(f"Saved: {out_png}")


# plot_pooled_and_exploratory(pooled_results, pertf_results)

/var/folders/c5/9gnt7c7x7x59ygrp90w3n60r0000gn/T/ipykernel_38693/525822854.py:79: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_ATAC_RNA_concordance_v2.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_ATAC_RNA_concordance_v2.png


In [65]:
def run_pooled_concordance_all_directions(tf_targets_by_ct, gene_atac, deg_all, deg_col_ct,
                                           n_perms=N_PERMUTATIONS, min_genes=MIN_GENES_PER_TEST_POOLED,
                                           seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    rows = []

    direction_specs = [
        ("KO-open / RNA-up",     "KO_open",   "up"),
        ("KO-closed / RNA-down", "KO_closed", "down"),
        ("KO-closed / RNA-up",   "KO_closed", "up"),    # discordant
        ("KO-open / RNA-down",   "KO_open",   "down"),  # discordant
    ]

    for ct_name, pooled_targets in tf_targets_by_ct.items():
        atac_sub = gene_atac[
            (gene_atac["cell_type"] == ct_name) & (gene_atac["gene"].isin(pooled_targets))
        ][["gene", "effect_direction"]]

        deg_sub = deg_all[
            (deg_all[deg_col_ct] == ct_name) & (deg_all["gene"].isin(pooled_targets)) & (deg_all["significant"] == True)
        ][["gene", "log2FC"]]

        merged = atac_sub.merge(deg_sub, on="gene", how="inner")
        if merged.shape[0] < min_genes:
            continue

        merged["rna_direction"] = np.where(merged["log2FC"] > 0, "up", "down")

        for direction_label, atac_dir, rna_dir in direction_specs:
            n_selected = merged.shape[0]
            other_atac_dir = "KO_closed" if atac_dir == "KO_open" else "KO_open"
            other_rna_dir = "down" if rna_dir == "up" else "up"

            a = int(((merged["effect_direction"] == atac_dir) & (merged["rna_direction"] == rna_dir)).sum())
            b = int(((merged["effect_direction"] == atac_dir) & (merged["rna_direction"] == other_rna_dir)).sum())
            c = int(((merged["effect_direction"] == other_atac_dir) & (merged["rna_direction"] == rna_dir)).sum())
            d = int(((merged["effect_direction"] == other_atac_dir) & (merged["rna_direction"] == other_rna_dir)).sum())

            table = np.array([[a, b], [c, d]])
            _, fisher_p = fisher_exact(table, alternative="two-sided")
            ci_table = Table2x2(table.astype(float) + 0.5)
            odds_ratio = ci_table.oddsratio
            ci_low, ci_high = ci_table.oddsratio_confint(alpha=0.05)

            atac_is_target_dir = (merged["effect_direction"] == atac_dir).to_numpy()
            rna_is_target_dir = (merged["rna_direction"] == rna_dir).to_numpy()
            n_rna_hits_observed = int(rna_is_target_dir.sum())
            n_genes = n_selected
            idx = np.arange(n_genes)

            perm_hits = np.empty(n_perms, dtype=int)
            for i in range(n_perms):
                perm_up_idx = rng_local.choice(idx, size=n_rna_hits_observed, replace=False)
                perm_mask = np.zeros(n_genes, dtype=bool)
                perm_mask[perm_up_idx] = True
                perm_hits[i] = int((atac_is_target_dir & perm_mask).sum())

            empirical_p = (np.sum(perm_hits >= a) + 1) / (n_perms + 1)

            rows.append({
                "cell_type": label_by_raw.get(ct_name, ct_name),
                "direction_label": direction_label,
                "concordance_type": "concordant" if direction_label in
                    ("KO-open / RNA-up", "KO-closed / RNA-down") else "discordant",
                "n_pooled_regulon_genes_tested": n_selected,
                "n_matching": a,
                "odds_ratio_descriptive": odds_ratio,
                "odds_ratio_95CI_low": ci_low,
                "odds_ratio_95CI_high": ci_high,
                "fisher_p_descriptive": fisher_p,
                "empirical_permutation_p": empirical_p,
                "expected_matching_permutation": perm_hits.mean(),
            })

    results_df = pd.DataFrame(rows)
    if results_df.empty:
        raise ValueError("No pooled cell-type tests met the minimum gene threshold.")

    results_df["permutation_FDR_BH"] = multipletests(
        results_df["empirical_permutation_p"].fillna(1.0), method="fdr_bh"
    )[1]
    results_df = results_df.sort_values("permutation_FDR_BH").reset_index(drop=True)
    return results_df


all_directions_results = run_pooled_concordance_all_directions(
    tf_targets_by_ct, gene_atac, deg_all, deg_col_ct
)

RESULTS_ALLDIR_CSV = os.path.join(OUT_DIR, "eRegulon_pooled_ATAC_RNA_all_directions_v3.csv")
all_directions_results.to_csv(RESULTS_ALLDIR_CSV, index=False)
print(f"Saved: {RESULTS_ALLDIR_CSV}")

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/eRegulon_pooled_ATAC_RNA_all_directions_v3.csv


In [32]:
# import matplotlib.pyplot as plt
# import matplotlib as mpl
# import numpy as np

# mpl.rcParams["font.family"] = "sans-serif"
# mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

# def plot_concordant_vs_discordant(all_dir_df, out_pdf, out_png):
#     df = all_dir_df.copy()
#     df["color"] = df["cell_type"].map(atac_colors).fillna("#999999")
#     df["significant"] = df["permutation_FDR_BH"] < 0.05
#     df["log2_OR"] = np.log2(df["odds_ratio_descriptive"].clip(lower=0.02))
#     df["log2_OR_CI_low"] = np.log2(df["odds_ratio_95CI_low"].clip(lower=0.02))
#     df["log2_OR_CI_high"] = np.log2(df["odds_ratio_95CI_high"].clip(lower=0.02))
#     df["label"] = df["cell_type"] + " (" + df["direction_label"] + ")"

#     concordant_df = df[df["concordance_type"] == "concordant"].sort_values("log2_OR").reset_index(drop=True)
#     discordant_df = df[df["concordance_type"] == "discordant"].sort_values("log2_OR").reset_index(drop=True)
#     concordant_df["y_pos"] = np.arange(len(concordant_df))
#     discordant_df["y_pos"] = np.arange(len(discordant_df))

#     n_rows = max(len(concordant_df), len(discordant_df))
#     fig_w_in = 14
#     fig_h_in = max(10, 0.62 * n_rows)

#     fig, (ax_left, ax_right) = plt.subplots(
#         1, 2, figsize=(fig_w_in, fig_h_in), sharex=True
#     )

#     for ax, sub_df, title in [
#         (ax_left, concordant_df, "Concordant\n(KO-open/RNA-up or KO-closed/RNA-down)"),
#         (ax_right, discordant_df, "Discordant\n(KO-closed/RNA-up or KO-open/RNA-down)"),
#     ]:
#         ax.errorbar(
#             sub_df["log2_OR"], sub_df["y_pos"],
#             xerr=[sub_df["log2_OR"] - sub_df["log2_OR_CI_low"], sub_df["log2_OR_CI_high"] - sub_df["log2_OR"]],
#             fmt="none", ecolor="#aaaaaa", elinewidth=0.9, capsize=3, capthick=0.9,
#             alpha=0.6, zorder=1
#         )
#         dot_size = np.where(sub_df["significant"], 220, 130)
#         ax.scatter(sub_df["log2_OR"], sub_df["y_pos"], s=dot_size, c=sub_df["color"],
#                    edgecolor="black", linewidth=1.0, zorder=3)
#         ax.axvline(0, color="black", linestyle="--", linewidth=1.2, zorder=0)
#         for _, row in sub_df[sub_df["significant"]].iterrows():
#             ax.text(row["log2_OR"] + 0.2, row["y_pos"], "*",
#                     fontsize=20, fontweight="bold", color="black", va="center")
#         ax.set_yticks(sub_df["y_pos"])
#         ax.set_yticklabels(sub_df["label"], fontsize=12, fontweight="bold")
#         for tick, ct in zip(ax.get_yticklabels(), sub_df["cell_type"]):
#             tick.set_color(atac_colors.get(ct, "#333333"))
#         ax.set_xlabel("log2(Odds ratio)", fontsize=13)
#         ax.set_title(title, fontsize=14, fontweight="bold")
#         ax.tick_params(axis="x", labelsize=11)
#         ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.4)
#         ax.spines[["top", "right"]].set_visible(False)

#     fig.suptitle(
#         "Pooled eRegulon-guided ATAC-RNA relationship: concordant vs discordant\n"
#         "All TFs' rho>0 targets pooled per cell type and direction, * FDR<0.05",
#         fontsize=15.5, fontweight="bold", y=1.02
#     )

#     plt.tight_layout()
#     fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
#     fig.savefig(out_png, bbox_inches="tight", dpi=300)
#     plt.close(fig)
#     print(f"Saved: {out_pdf}")
#     print(f"Saved: {out_png}")


# FIG_ALLDIR_PDF = os.path.join(OUT_DIR, "Fig_eRegulon_pooled_concordant_vs_discordant_v4.pdf")
# FIG_ALLDIR_PNG = FIG_ALLDIR_PDF.replace(".pdf", ".png")
# plot_concordant_vs_discordant(all_directions_results, FIG_ALLDIR_PDF, FIG_ALLDIR_PNG)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_concordant_vs_discordant_v4.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_concordant_vs_discordant_v4.png


In [33]:
# import matplotlib.pyplot as plt
# import matplotlib as mpl
# import numpy as np

# mpl.rcParams["font.family"] = "sans-serif"
# mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

# def plot_single_direction_panel(sub_df, title, out_pdf, out_png):
#     sub_df = sub_df.sort_values("log2_OR").reset_index(drop=True)
#     sub_df["y_pos"] = np.arange(len(sub_df))

#     n_rows = len(sub_df)
#     fig_w_in = 9
#     fig_h_in = max(8, 0.62 * n_rows)

#     fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in))

#     # Dot size now maps to n_pooled_regulon_genes_tested -- the sample size
#     # driving statistical power for each test, not a binary significance flag
#     n_genes = sub_df["n_pooled_regulon_genes_tested"]
#     size_min, size_max = 60, 320
#     n_min, n_max = n_genes.min(), n_genes.max()
#     if n_max > n_min:
#         dot_size = size_min + (n_genes - n_min) / (n_max - n_min) * (size_max - size_min)
#     else:
#         dot_size = np.full(len(sub_df), (size_min + size_max) / 2)

#     ax.errorbar(
#         sub_df["log2_OR"], sub_df["y_pos"],
#         xerr=[sub_df["log2_OR"] - sub_df["log2_OR_CI_low"], sub_df["log2_OR_CI_high"] - sub_df["log2_OR"]],
#         fmt="none", ecolor="#aaaaaa", elinewidth=0.9, capsize=3, capthick=0.9,
#         alpha=0.6, zorder=1
#     )
#     ax.scatter(sub_df["log2_OR"], sub_df["y_pos"], s=dot_size, c=sub_df["color"],
#                edgecolor="black", linewidth=1.0, zorder=3)
#     ax.axvline(0, color="black", linestyle="--", linewidth=1.2, zorder=0)
#     for _, row in sub_df[sub_df["significant"]].iterrows():
#         ax.text(row["log2_OR"] + 0.2, row["y_pos"], "*",
#                 fontsize=20, fontweight="bold", color="black", va="center")
#     ax.set_yticks(sub_df["y_pos"])
#     ax.set_yticklabels(sub_df["label"], fontsize=12, fontweight="bold")
#     for tick, ct in zip(ax.get_yticklabels(), sub_df["cell_type"]):
#         tick.set_color(atac_colors.get(ct, "#333333"))
#     ax.set_xlabel("log2(Odds ratio)", fontsize=13)
#     ax.set_title(title, fontsize=15, fontweight="bold", pad=14)
#     ax.tick_params(axis="x", labelsize=11)
#     ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.4)
#     ax.spines[["top", "right"]].set_visible(False)

#     # Size legend: pick 3-4 representative gene counts across the observed range
#     legend_n_values = sorted(set(np.linspace(n_min, n_max, 4).round().astype(int)))
#     legend_sizes = [
#         size_min + (v - n_min) / (n_max - n_min) * (size_max - size_min) if n_max > n_min
#         else (size_min + size_max) / 2
#         for v in legend_n_values
#     ]
#     legend_handles = [
#         plt.scatter([], [], s=s, c="#999999", edgecolor="black", linewidth=1.0, label=f"n={v}")
#         for v, s in zip(legend_n_values, legend_sizes)
#     ]
#     ax.legend(
#         handles=legend_handles, title="Pooled regulon\ngenes tested",
#         loc="lower right", fontsize=9.5, title_fontsize=10,
#         frameon=True, framealpha=0.9, borderpad=1.0, labelspacing=1.3
#     )

#     plt.tight_layout()
#     fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
#     fig.savefig(out_png, bbox_inches="tight", dpi=300)
#     plt.close(fig)
#     print(f"Saved: {out_pdf}")
#     print(f"Saved: {out_png}")


# def plot_concordant_vs_discordant_separate(all_dir_df, out_dir=OUT_DIR):
#     df = all_dir_df.copy()
#     df["color"] = df["cell_type"].map(atac_colors).fillna("#999999")
#     df["significant"] = df["permutation_FDR_BH"] < 0.05
#     df["log2_OR"] = np.log2(df["odds_ratio_descriptive"].clip(lower=0.02))
#     df["log2_OR_CI_low"] = np.log2(df["odds_ratio_95CI_low"].clip(lower=0.02))
#     df["log2_OR_CI_high"] = np.log2(df["odds_ratio_95CI_high"].clip(lower=0.02))
#     df["label"] = df["cell_type"] + " (" + df["direction_label"] + ")"

#     concordant_df = df[df["concordance_type"] == "concordant"].copy()
#     discordant_df = df[df["concordance_type"] == "discordant"].copy()

#     plot_single_direction_panel(
#         concordant_df,
#         "Concordant ATAC-RNA relationship\n(KO-open/RNA-up or KO-closed/RNA-down), * FDR<0.05",
#         os.path.join(out_dir, "Fig_eRegulon_pooled_concordant_v4.pdf"),
#         os.path.join(out_dir, "Fig_eRegulon_pooled_concordant_v4.png"),
#     )
#     plot_single_direction_panel(
#         discordant_df,
#         "Discordant ATAC-RNA relationship\n(KO-closed/RNA-up or KO-open/RNA-down), * FDR<0.05",
#         os.path.join(out_dir, "Fig_eRegulon_pooled_discordant_v4.pdf"),
#         os.path.join(out_dir, "Fig_eRegulon_pooled_discordant_v4.png"),
#     )


# plot_concordant_vs_discordant_separate(all_directions_results)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_concordant_v4.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_concordant_v4.png
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_discordant_v4.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_eRegulon_pooled_discordant_v4.png


In [67]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from statsmodels.stats.multitest import multipletests

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

BASE = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus"
FURTHER_OUT = os.path.join(BASE, "scenicplus_further_analysis")
FIG_OUT = os.path.join(FURTHER_OUT, "manuscript_figures")
CSV_DIR = os.path.join(FURTHER_OUT, "TF_target_csvs_multi_spec")
DEG_CSV = os.path.join(FURTHER_OUT, "DEG_KO_vs_Ctrl", "DEG_all_celltypes_KO_vs_Ctrl.csv")
OUT_DIR = os.path.join(FIG_OUT, "cross_modal_ATAC_RNA_concordance")
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
N_PERMUTATIONS = 10000
MIN_GENES_PER_TEST = 15
rng = np.random.default_rng(RANDOM_SEED)

# Collapsed cell-type list -- no separate perineuronal/myelinating oligodendrocytes,
# both folded into a single "Oligodendrocytes" category, which takes E8A0BF
# (this was previously E8A0BF's original slot in the full 22-category reference map)
celltype_order = [
    "Layer 2/3 IT neurons", "Layer 4 sensory neurons", "Layer 5a IT neurons",
    "Layer 5b PT neurons", "Layer 5/6 IT neurons", "Layer 6a corticothalamic neurons",
    "Layer 6b neurons", "Deep-layer extratelencephalic neurons",
    "Corticospinal neurons Type I", "Corticospinal neurons Type II",
    "PV+ interneurons", "SST+ interneurons", "VIP+ interneurons",
    "Astrocytes", "Oligodendrocyte precursor cells", "Oligodendrocytes",
    "Microglia", "Endothelial cells", "Leptomeningeal cells", "Meningeal fibroblasts"
]
hex_colors = [
    "6B5B95", "45B8AC", "955251", "4E84C4", "B565A7", "88B04B", "7B6888",
    "C3447A", "009B77", "EFC050", "7FCDCD", "DD4124", "5B5EA6", "E07A5F",
    "4BACC6", "E8A0BF", "C17BAE", "DECF3F", "789262", "BC243C"
]
atac_colors = {ct: f"#{h}" for ct, h in zip(celltype_order, hex_colors)}
raw_celltype_names = [ct.replace("+", "") for ct in celltype_order]
label_by_raw = {raw.strip(): label for raw, label in zip(raw_celltype_names, celltype_order)}

# CellChat-style abbreviations for y-axis labels, keyed by full cell-type name
abbrev_map = {
    "Layer 2/3 IT neurons": "L2/3 IT",
    "Layer 4 sensory neurons": "L4 Sensory",
    "Layer 5a IT neurons": "L5a IT",
    "Layer 5b PT neurons": "L5b PT",
    "Layer 5/6 IT neurons": "L5/6 IT",
    "Layer 6a corticothalamic neurons": "L6a CT",
    "Layer 6b neurons": "L6b",
    "Deep-layer extratelencephalic neurons": "Deep ET",
    "Corticospinal neurons Type I": "CSN Type I",
    "Corticospinal neurons Type II": "CSN Type II",
    "PV+ interneurons": "PV+ Int",
    "SST+ interneurons": "SST+ Int",
    "VIP+ interneurons": "VIP+ Int",
    "Astrocytes": "Astrocyte",
    "Oligodendrocyte precursor cells": "OPC",
    "Oligodendrocytes": "OL",
    "Microglia": "Microglia",
    "Endothelial cells": "Endothelial",
    "Leptomeningeal cells": "Leptomeningeal FB",
    "Meningeal fibroblasts": "Meningeal FB",
}


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9-]", "_", s)


safe_to_raw = {safe_name(raw): raw for raw in raw_celltype_names}
csv_files = []
for path in glob.glob(os.path.join(CSV_DIR, "*.csv")):
    fname = os.path.basename(path)[:-4]
    for safe_ct, raw_ct in safe_to_raw.items():
        if fname.endswith(safe_ct):
            tf_name = fname[: -(len(safe_ct) + 1)]
            csv_files.append((tf_name, raw_ct, path))
            break

deg_all = pd.read_csv(DEG_CSV)
deg_all["gene"] = deg_all["gene"].astype(str)
deg_col_ct = "celltype" if "celltype" in deg_all.columns else "cell_type"
print("Cell types in deg_all:", sorted(deg_all[deg_col_ct].unique()))

tf_targets_by_ct = {}
for tf_name, ct_name, csv_path in csv_files:
    tf_df = pd.read_csv(csv_path)
    if "rho" not in tf_df.columns or "target" not in tf_df.columns:
        continue
    targets = set(tf_df[tf_df["rho"] > 0]["target"].dropna().astype(str).str.strip())
    if targets:
        tf_targets_by_ct.setdefault(ct_name, set()).update(targets)

# -----------------------------------------------------------------------------
# PANEL A: target-overlap enrichment (RNA only, pooled per cell type)
# FIX: filter deg_all using canonical label (with "+"), not stripped ct_name
# -----------------------------------------------------------------------------

def run_overlap_enrichment(tf_targets_by_ct, deg_all, deg_col_ct, label_by_raw,
                            n_perms=N_PERMUTATIONS, min_genes=MIN_GENES_PER_TEST, seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    rows = []
    for ct_name, pooled_targets in tf_targets_by_ct.items():
        canonical_ct = label_by_raw.get(ct_name, ct_name)
        ct_deg_table = deg_all[deg_all[deg_col_ct] == canonical_ct]
        background = set(ct_deg_table["gene"].astype(str))
        targets_in_bg = pooled_targets & background
        if len(targets_in_bg) < min_genes:
            print(f"Skipping {canonical_ct}: only {len(targets_in_bg)} targets in background (min {min_genes})")
            continue

        sig_degs = set(ct_deg_table[ct_deg_table["significant"] == True]["gene"].astype(str))
        n_degs = len(sig_degs & background)
        if n_degs == 0:
            print(f"Skipping {canonical_ct}: no significant DEGs in background")
            continue

        observed = len(targets_in_bg & sig_degs)
        bg_arr = np.array(list(background))
        n_bg = len(bg_arr)
        idx = np.arange(n_bg)
        targets_mask = np.isin(bg_arr, list(targets_in_bg))

        null_overlap = np.empty(n_perms, dtype=int)
        for i in range(n_perms):
            perm_deg_idx = rng_local.choice(idx, size=n_degs, replace=False)
            perm_mask = np.zeros(n_bg, dtype=bool)
            perm_mask[perm_deg_idx] = True
            null_overlap[i] = int((targets_mask & perm_mask).sum())

        p_val = (np.sum(null_overlap >= observed) + 1) / (n_perms + 1)
        expected = null_overlap.mean()
        enrichment_ratio = observed / expected if expected > 0 else np.nan

        rows.append({
            "cell_type": canonical_ct,
            "n_targets_tested": len(targets_in_bg),
            "n_DEGs_background": n_degs,
            "n_overlap_observed": observed,
            "expected_overlap": expected,
            "enrichment_ratio": enrichment_ratio,
            "empirical_permutation_p": p_val,
        })

    df = pd.DataFrame(rows)
    df["permutation_FDR_BH"] = multipletests(df["empirical_permutation_p"].fillna(1.0), method="fdr_bh")[1]
    return df.sort_values("permutation_FDR_BH").reset_index(drop=True)


overlap_results = run_overlap_enrichment(tf_targets_by_ct, deg_all, deg_col_ct, label_by_raw)
overlap_results.to_csv(os.path.join(OUT_DIR, "TF_target_DEG_overlap_pooled_v5.csv"), index=False)
print(overlap_results[["cell_type", "n_targets_tested", "n_DEGs_background",
                        "enrichment_ratio", "empirical_permutation_p", "permutation_FDR_BH"]].to_string(index=False))

# # -----------------------------------------------------------------------------
# # PANEL B: cross-modal directional concordance (reuse existing pooled_results)
# # -----------------------------------------------------------------------------

# concordance_results = pooled_results.copy()
# concordance_results = concordance_results[
#     concordance_results["direction_label"].isin(["KO-open / RNA-up", "KO-closed / RNA-down"])
# ].copy()

# # -----------------------------------------------------------------------------
# # Combined figure, fixed aspect ratio, fixed margins, abbreviated labels
# # -----------------------------------------------------------------------------

# def plot_two_panel_fixed_layout(overlap_df, concordance_df, out_pdf, out_png):
#     ASPECT_W, ASPECT_H = 300, 760
#     fig_w_in = 9.0
#     fig_h_in = fig_w_in * (ASPECT_H / ASPECT_W)

#     fig = plt.figure(figsize=(fig_w_in, fig_h_in))
#     # Shorter labels -> smaller reserved margin, freeing width for bigger fonts
#     LEFT_MARGIN = 0.24
#     RIGHT_MARGIN = 0.97
#     TOP_MARGIN = 0.93
#     BOTTOM_MARGIN = 0.07
#     GAP = 0.07

#     gs = fig.add_gridspec(
#         2, 1, left=LEFT_MARGIN, right=RIGHT_MARGIN, top=TOP_MARGIN, bottom=BOTTOM_MARGIN,
#         hspace=GAP, height_ratios=[1, 1]
#     )

#     # --- Panel A: overlap enrichment ---
#     ax_a = fig.add_subplot(gs[0])
#     df_a = overlap_df.copy()
#     df_a["color"] = df_a["cell_type"].map(atac_colors).fillna("#999999")
#     df_a["significant"] = df_a["permutation_FDR_BH"] < 0.05
#     df_a["log2_enrichment"] = np.log2(df_a["enrichment_ratio"].clip(lower=0.05))
#     df_a["abbrev"] = df_a["cell_type"].map(abbrev_map).fillna(df_a["cell_type"])
#     df_a = df_a.sort_values("log2_enrichment").reset_index(drop=True)
#     df_a["y_pos"] = np.arange(len(df_a))

#     dot_size_a = np.where(df_a["significant"], 130, 80)
#     ax_a.scatter(df_a["log2_enrichment"], df_a["y_pos"], s=dot_size_a, c=df_a["color"],
#                  edgecolor="black", linewidth=0.8, zorder=3)
#     ax_a.axvline(0, color="black", linestyle="--", linewidth=1.0, zorder=0)
#     for _, row in df_a[df_a["significant"]].iterrows():
#         ax_a.text(row["log2_enrichment"] + 0.15, row["y_pos"], "*",
#                    fontsize=15, fontweight="bold", va="center")
#     ax_a.set_yticks(df_a["y_pos"])
#     ax_a.set_yticklabels(df_a["abbrev"], fontsize=11, fontweight="bold")
#     for tick, ct in zip(ax_a.get_yticklabels(), df_a["cell_type"]):
#         tick.set_color(atac_colors.get(ct, "#333333"))
#     ax_a.set_xlabel("log2(Enrichment ratio), TF targets among DEGs", fontsize=10.5)
#     ax_a.set_title("A. Target-overlap enrichment (RNA only)\nPooled TF targets vs KO/Ctrl DEGs, * FDR<0.05",
#                     fontsize=12, fontweight="bold")
#     ax_a.tick_params(axis="x", labelsize=10)
#     ax_a.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.4)
#     ax_a.spines[["top", "right"]].set_visible(False)

#     # --- Panel B: cross-modal directional concordance ---
#     ax_b = fig.add_subplot(gs[1])
#     df_b = concordance_df.copy()
#     df_b["color"] = df_b["cell_type"].map(atac_colors).fillna("#999999")
#     df_b["significant"] = df_b["permutation_FDR_BH"] < 0.05
#     df_b["log2_OR"] = np.log2(df_b["odds_ratio_descriptive"].clip(lower=0.02))
#     df_b["log2_OR_CI_low"] = np.log2(df_b["odds_ratio_95CI_low"].clip(lower=0.02))
#     df_b["log2_OR_CI_high"] = np.log2(df_b["odds_ratio_95CI_high"].clip(lower=0.02))
#     df_b["abbrev"] = df_b["cell_type"].map(abbrev_map).fillna(df_b["cell_type"])
#     dir_short = df_b["direction_label"].map({
#         "KO-open / RNA-up": "open/up", "KO-closed / RNA-down": "closed/down"
#     })
#     df_b["label"] = df_b["abbrev"] + " (" + dir_short + ")"
#     df_b = df_b.sort_values("log2_OR").reset_index(drop=True)
#     df_b["y_pos"] = np.arange(len(df_b))

#     ax_b.errorbar(
#         df_b["log2_OR"], df_b["y_pos"],
#         xerr=[df_b["log2_OR"] - df_b["log2_OR_CI_low"], df_b["log2_OR_CI_high"] - df_b["log2_OR"]],
#         fmt="none", ecolor="#aaaaaa", elinewidth=0.8, capsize=2.5, capthick=0.8, alpha=0.6, zorder=1
#     )
#     dot_size_b = np.where(df_b["significant"], 130, 80)
#     ax_b.scatter(df_b["log2_OR"], df_b["y_pos"], s=dot_size_b, c=df_b["color"],
#                  edgecolor="black", linewidth=0.8, zorder=3)
#     ax_b.axvline(0, color="black", linestyle="--", linewidth=1.0, zorder=0)
#     for _, row in df_b[df_b["significant"]].iterrows():
#         ax_b.text(row["log2_OR"] + 0.15, row["y_pos"], "*",
#                    fontsize=15, fontweight="bold", va="center")
#     ax_b.set_yticks(df_b["y_pos"])
#     ax_b.set_yticklabels(df_b["label"], fontsize=10.5, fontweight="bold")
#     for tick, ct in zip(ax_b.get_yticklabels(), df_b["cell_type"]):
#         tick.set_color(atac_colors.get(ct, "#333333"))
#     ax_b.set_xlabel("log2(Odds ratio), ATAC direction vs RNA direction", fontsize=10.5)
#     ax_b.set_title("B. Cross-modal directional concordance (ATAC + RNA)\nPooled regulon targets, * FDR<0.05",
#                     fontsize=12, fontweight="bold")
#     ax_b.tick_params(axis="x", labelsize=10)
#     ax_b.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.4)
#     ax_b.spines[["top", "right"]].set_visible(False)

#     fig.suptitle("Two complementary tests of TF regulon activity in KO vs Ctrl",
#                  fontsize=13, fontweight="bold", y=0.985)

#     fig.savefig(out_pdf, bbox_inches=None, dpi=300)
#     fig.savefig(out_png, bbox_inches=None, dpi=300)
#     plt.close(fig)
#     print(f"Saved: {out_pdf}")
#     print(f"Saved: {out_png}")


# FIG_PDF_V5 = os.path.join(OUT_DIR, "Fig_TF_overlap_vs_crossmodal_concordance_v5.pdf")
# FIG_PNG_V5 = FIG_PDF_V5.replace(".pdf", ".png")
# plot_two_panel_fixed_layout(overlap_results, concordance_results, FIG_PDF_V5, FIG_PNG_V5)

Cell types in deg_all: ['Astrocytes', 'Corticospinal neurons (Type I)', 'Deep-layer extratelencephalic neurons', 'Endothelial cells', 'Layer 2/3 IT neurons', 'Layer 4 sensory neurons', 'Layer 5/6 IT neurons', 'Layer 5a IT neurons', 'Layer 5b PT neurons', 'Layer 6a corticothalamic neurons', 'Layer 6b neurons', 'Leptomeningeal cells', 'Meningeal fibroblasts', 'Microglia', 'Oligodendrocyte precursor cells', 'Oligodendrocytes', 'PV+ interneurons', 'SST+ interneurons', 'VIP+ interneurons']
Skipping Meningeal fibroblasts: no significant DEGs in background
                            cell_type  n_targets_tested  n_DEGs_background  enrichment_ratio  empirical_permutation_p  permutation_FDR_BH
                 Layer 2/3 IT neurons                67                873          2.192469                 0.000100            0.000178
              Layer 4 sensory neurons                91               1735          1.145941                 0.000100            0.000178
                     PV+ inter

In [72]:
print("pooled_results shape:", pooled_results.shape)
print(pooled_results[pooled_results["cell_type"].str.contains("PV")])

concordance_results = pooled_results.copy()

print("concordance_results shape after reassignment:", concordance_results.shape)
print(concordance_results[concordance_results["cell_type"].str.contains("PV")])

pooled_results shape: (20, 14)
           cell_type       direction_label  n_pooled_regulon_genes_tested  \
14  PV+ interneurons  KO-closed / RNA-down                             18   
16  PV+ interneurons      KO-open / RNA-up                             18   

    n_concordant  n_discordant_atacdir_rna_other  \
14             9                               2   
16             1                               6   

    n_discordant_other_atacdir_rnadir  n_neither  odds_ratio_descriptive  \
14                                  6          1                0.876923   
16                                  2          9                0.876923   

    odds_ratio_95CI_low  odds_ratio_95CI_high  fisher_p_descriptive  \
14             0.091805              8.376352                   1.0   
16             0.091805              8.376352                   1.0   

    empirical_permutation_p  expected_concordant_permutation  \
14                  0.80292                           9.1632   
16       

In [73]:
def plot_two_panel_fixed_layout(overlap_df, concordance_df, out_pdf, out_png):
    INCHES_PER_ROW = 0.34
    fig_w_in = 9.0

    df_a_prep = overlap_df.copy()
    n_rows_a = len(df_a_prep)

    df_b_prep = concordance_df[concordance_df["direction_label"] == "KO-open / RNA-up"].copy()
    n_rows_b = len(df_b_prep)

    panel_a_h = n_rows_a * INCHES_PER_ROW
    panel_b_h = n_rows_b * INCHES_PER_ROW
    top_gap_in = 0.75      # space for suptitle above panel A
    mid_gap_in = 1.7        # space for panel A's x-axis label + panel B's title stacked
    bottom_gap_in = 0.55    # space for panel B's x-axis label
    fig_h_in = top_gap_in + panel_a_h + mid_gap_in + panel_b_h + bottom_gap_in

    fig = plt.figure(figsize=(fig_w_in, fig_h_in))
    LEFT_MARGIN = 0.24
    RIGHT_MARGIN = 0.5
    TOP_MARGIN = 1 - top_gap_in / fig_h_in
    BOTTOM_MARGIN = bottom_gap_in / fig_h_in
    GAP = mid_gap_in / panel_a_h  # hspace is relative to average axes height in gridspec

    gs = fig.add_gridspec(
        2, 1, left=LEFT_MARGIN, right=RIGHT_MARGIN, top=TOP_MARGIN, bottom=BOTTOM_MARGIN,
        hspace=GAP, height_ratios=[panel_a_h, panel_b_h]
    )

    # --- Panel A: overlap enrichment ---
    ax_a = fig.add_subplot(gs[0])
    df_a = df_a_prep
    df_a["color"] = df_a["cell_type"].map(atac_colors).fillna("#999999")
    df_a["significant"] = df_a["permutation_FDR_BH"] < 0.05
    df_a["log2_enrichment"] = np.log2(df_a["enrichment_ratio"].clip(lower=0.05))
    df_a["abbrev"] = df_a["cell_type"].map(abbrev_map).fillna(df_a["cell_type"])
    df_a = df_a.sort_values("log2_enrichment").reset_index(drop=True)
    df_a["y_pos"] = np.arange(len(df_a))

    dot_size_a = np.where(df_a["significant"], 130, 80)
    ax_a.scatter(df_a["log2_enrichment"], df_a["y_pos"], s=dot_size_a, c=df_a["color"],
                 edgecolor="black", linewidth=0.8, zorder=3)
    ax_a.axvline(0, color="black", linestyle="--", linewidth=1.0, zorder=0)
    for _, row in df_a[df_a["significant"]].iterrows():
        ax_a.text(row["log2_enrichment"] + 0.15, row["y_pos"], "*",
                   fontsize=15, fontweight="bold", va="center")
    ax_a.set_yticks(df_a["y_pos"])
    ax_a.set_yticklabels(df_a["abbrev"], fontsize=11, fontweight="bold")
    for tick, ct in zip(ax_a.get_yticklabels(), df_a["cell_type"]):
        tick.set_color(atac_colors.get(ct, "#333333"))
    ax_a.set_ylim(-0.7, df_a["y_pos"].max() + 0.7)
    ax_a.set_xlabel("log2(Enrichment ratio), TF targets among DEGs", fontsize=10.5)
    ax_a.set_title("A. Target-overlap enrichment (RNA only)\nPooled TF targets vs KO/Ctrl DEGs, * FDR<0.05",
                    fontsize=12, fontweight="bold", pad=10)
    ax_a.tick_params(axis="x", labelsize=10)
    ax_a.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.4)
    ax_a.spines[["top", "right"]].set_visible(False)

    # --- Panel B: cross-modal concordance, one row per cell type ---
    ax_b = fig.add_subplot(gs[1])
    df_b = df_b_prep
    df_b["color"] = df_b["cell_type"].map(atac_colors).fillna("#999999")
    df_b["significant"] = df_b["permutation_FDR_BH"] < 0.05
    df_b["log2_OR"] = np.log2(df_b["odds_ratio_descriptive"].clip(lower=0.02))
    df_b["log2_OR_CI_low"] = np.log2(df_b["odds_ratio_95CI_low"].clip(lower=0.02))
    df_b["log2_OR_CI_high"] = np.log2(df_b["odds_ratio_95CI_high"].clip(lower=0.02))
    df_b["abbrev"] = df_b["cell_type"].map(abbrev_map).fillna(df_b["cell_type"])
    df_b = df_b.sort_values("log2_OR").reset_index(drop=True)
    df_b["y_pos"] = np.arange(len(df_b))

    ax_b.errorbar(
        df_b["log2_OR"], df_b["y_pos"],
        xerr=[df_b["log2_OR"] - df_b["log2_OR_CI_low"], df_b["log2_OR_CI_high"] - df_b["log2_OR"]],
        fmt="none", ecolor="#aaaaaa", elinewidth=0.8, capsize=2.5, capthick=0.8, alpha=0.6, zorder=1
    )
    dot_size_b = np.where(df_b["significant"], 130, 80)
    ax_b.scatter(df_b["log2_OR"], df_b["y_pos"], s=dot_size_b, c=df_b["color"],
                 edgecolor="black", linewidth=0.8, zorder=3)
    ax_b.axvline(0, color="black", linestyle="--", linewidth=1.0, zorder=0)
    for _, row in df_b[df_b["significant"]].iterrows():
        ax_b.text(row["log2_OR"] + 0.15, row["y_pos"], "*",
                   fontsize=15, fontweight="bold", va="center")
    ax_b.set_yticks(df_b["y_pos"])
    ax_b.set_yticklabels(df_b["abbrev"], fontsize=11, fontweight="bold")
    for tick, ct in zip(ax_b.get_yticklabels(), df_b["cell_type"]):
        tick.set_color(atac_colors.get(ct, "#333333"))
    ax_b.set_ylim(-0.7, df_b["y_pos"].max() + 0.7)
    ax_b.set_xlabel("log2(Odds ratio), ATAC direction vs RNA direction", fontsize=10.5)
    ax_b.set_title("B. Cross-modal directional concordance (ATAC + RNA)\nPooled regulon targets, * FDR<0.05",
                    fontsize=12, fontweight="bold", pad=10)
    ax_b.tick_params(axis="x", labelsize=10)
    ax_b.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.4)
    ax_b.spines[["top", "right"]].set_visible(False)

    # Legend: upper-left corner, no frame/box
    sig_handle = plt.scatter([], [], s=130, c="#999999", edgecolor="black", linewidth=0.8, label="FDR < 0.05")
    nonsig_handle = plt.scatter([], [], s=80, c="#999999", edgecolor="black", linewidth=0.8, label="Not significant")
    ci_handle = plt.Line2D([], [], color="#aaaaaa", linewidth=1.5, alpha=0.6, label="95% CI")
    ax_b.legend(
        handles=[sig_handle, nonsig_handle, ci_handle],
        loc="upper left", fontsize=8.5, frameon=False,
        borderpad=0.6, labelspacing=0.7, handletextpad=0.7
    )

    fig.suptitle("Two complementary tests of TF regulon activity in KO vs Ctrl",
                  fontsize=13, fontweight="bold", y=1 - (0.25 / fig_h_in))

    fig.savefig(out_pdf, bbox_inches=None, dpi=300)
    fig.savefig(out_png, bbox_inches=None, dpi=300)
    plt.close(fig)
    print(f"Saved: {out_pdf}")
    print(f"Saved: {out_png}")


FIG_PDF_V8 = os.path.join(OUT_DIR, "Fig_TF_overlap_vs_crossmodal_concordance_v8.pdf")
FIG_PNG_V8 = FIG_PDF_V8.replace(".pdf", ".png")
plot_two_panel_fixed_layout(overlap_results, concordance_results, FIG_PDF_V8, FIG_PNG_V8)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_TF_overlap_vs_crossmodal_concordance_v8.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_TF_overlap_vs_crossmodal_concordance_v8.png


In [69]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

def plot_permutation_null_distributions(perm_results_dict, observed_dict, fdr_dict,
                                          abbrev_map, atac_colors, out_pdf, out_png,
                                          n_cols=3):
    """
    perm_results_dict: {cell_type_raw: np.array of permuted overlap/OR values, length n_perms}
    observed_dict: {cell_type_raw: observed statistic value}
    fdr_dict: {cell_type_raw: permutation_FDR_BH value}
    """
    cell_types = list(perm_results_dict.keys())
    n_plots = len(cell_types)
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.0 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, ct in enumerate(cell_types):
        ax = axes[i]
        null_vals = perm_results_dict[ct]
        observed = observed_dict[ct]
        fdr = fdr_dict[ct]
        color = atac_colors.get(ct, "#999999")
        label = abbrev_map.get(ct, ct)

        ax.hist(null_vals, bins=30, color=color, alpha=0.5, edgecolor="white", linewidth=0.3)
        ax.axvline(observed, color="black", linestyle="--", linewidth=1.6, zorder=3)

        sig_str = "FDR<0.05 *" if fdr < 0.05 else "n.s."
        ax.text(0.97, 0.92, sig_str, transform=ax.transAxes, ha="right", va="top",
                fontsize=8.5, fontweight="bold", color="black" if fdr < 0.05 else "#777777")

        ax.set_title(label, fontsize=10.5, fontweight="bold", color=color)
        ax.set_xlabel("Overlap count (null)", fontsize=8)
        ax.set_ylabel("Frequency", fontsize=8)
        ax.tick_params(labelsize=7.5)
        ax.spines[["top", "right"]].set_visible(False)

    for j in range(n_plots, len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        "Permutation null distributions vs observed overlap\n"
        "Dashed line = observed value, distribution = permuted null (n=10,000 shuffles per cell type)",
        fontsize=12.5, fontweight="bold", y=1.02
    )

    plt.tight_layout()
    fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
    fig.savefig(out_png, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {out_pdf}")
    print(f"Saved: {out_png}")

In [76]:
import os
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

def run_overlap_enrichment_with_cache(tf_targets_by_ct, deg_all, deg_col_ct, label_by_raw,
                                        n_perms=N_PERMUTATIONS, min_genes=MIN_GENES_PER_TEST, seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    rows = []
    null_dist_cache = {}     # keyed by raw_ct (stripped), for consistency with downstream plots
    observed_cache = {}

    for ct_name, pooled_targets in tf_targets_by_ct.items():
        canonical_ct = label_by_raw.get(ct_name, ct_name)  # e.g. "PV+ interneurons"
        ct_deg_table = deg_all[deg_all[deg_col_ct] == canonical_ct]
        background = set(ct_deg_table["gene"].astype(str))
        targets_in_bg = pooled_targets & background
        if len(targets_in_bg) < min_genes:
            print(f"Skipping {canonical_ct}: only {len(targets_in_bg)} targets in background (min {min_genes})")
            continue

        sig_degs = set(ct_deg_table[ct_deg_table["significant"] == True]["gene"].astype(str))
        n_degs = len(sig_degs & background)
        if n_degs == 0:
            print(f"Skipping {canonical_ct}: no significant DEGs in background")
            continue

        observed = len(targets_in_bg & sig_degs)
        bg_arr = np.array(list(background))
        n_bg = len(bg_arr)
        idx = np.arange(n_bg)
        targets_mask = np.isin(bg_arr, list(targets_in_bg))

        null_overlap = np.empty(n_perms, dtype=int)
        for i in range(n_perms):
            perm_deg_idx = rng_local.choice(idx, size=n_degs, replace=False)
            perm_mask = np.zeros(n_bg, dtype=bool)
            perm_mask[perm_deg_idx] = True
            null_overlap[i] = int((targets_mask & perm_mask).sum())

        p_val = (np.sum(null_overlap >= observed) + 1) / (n_perms + 1)
        expected = null_overlap.mean()
        enrichment_ratio = observed / expected if expected > 0 else np.nan

        null_dist_cache[ct_name] = null_overlap    # raw_ct key, matches df["raw_ct"] downstream
        observed_cache[ct_name] = observed

        rows.append({
            "cell_type": canonical_ct,
            "n_targets_tested": len(targets_in_bg),
            "n_DEGs_background": n_degs,
            "n_overlap_observed": observed,
            "expected_overlap": expected,
            "enrichment_ratio": enrichment_ratio,
            "empirical_permutation_p": p_val,
        })

    df = pd.DataFrame(rows)
    df["permutation_FDR_BH"] = multipletests(df["empirical_permutation_p"].fillna(1.0), method="fdr_bh")[1]
    df = df.sort_values("permutation_FDR_BH").reset_index(drop=True)

    return df, null_dist_cache, observed_cache


overlap_results, null_dist_cache, observed_cache = run_overlap_enrichment_with_cache(
    tf_targets_by_ct, deg_all, deg_col_ct, label_by_raw
)

# Simpler, robust FDR lookup keyed by raw cell type name
raw_to_label = label_by_raw
label_to_fdr = dict(zip(overlap_results["cell_type"], overlap_results["permutation_FDR_BH"]))
fdr_dict = {ct: label_to_fdr.get(raw_to_label.get(ct, ct), 1.0) for ct in null_dist_cache}

os.makedirs(OUT_DIR, exist_ok=True)
np.savez(os.path.join(OUT_DIR, "null_distributions_cache.npz"), **null_dist_cache)
print(f"Saved: {os.path.join(OUT_DIR, 'null_distributions_cache.npz')}")

FIG_PERM_PDF = os.path.join(OUT_DIR, "Fig_permutation_null_distributions.pdf")
FIG_PERM_PNG = FIG_PERM_PDF.replace(".pdf", ".png")

plot_permutation_null_distributions(
    perm_results_dict=null_dist_cache,
    observed_dict=observed_cache,
    fdr_dict=fdr_dict,
    abbrev_map=abbrev_map,
    atac_colors=atac_colors,
    out_pdf=FIG_PERM_PDF,
    out_png=FIG_PERM_PNG,
    n_cols=3
)

Skipping Meningeal fibroblasts: no significant DEGs in background
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/null_distributions_cache.npz
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_permutation_null_distributions.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_permutation_null_distributions.png


In [77]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.stats import gaussian_kde

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

def plot_forest_with_ridgelines(overlap_df, null_dist_cache, observed_cache,
                                  abbrev_map, atac_colors, out_pdf, out_png,
                                  ridge_height=0.8):
    df = overlap_df.copy()
    df["color"] = df["cell_type"].map(atac_colors).fillna("#999999")
    df["significant"] = df["permutation_FDR_BH"] < 0.05
    df["log2_enrichment"] = np.log2(df["enrichment_ratio"].clip(lower=0.05))
    df["abbrev"] = df["cell_type"].map(abbrev_map).fillna(df["cell_type"])
    df["raw_ct"] = df["cell_type"].map({v: k for k, v in label_by_raw.items()})

    df = df.sort_values("log2_enrichment").reset_index(drop=True)
    df["y_pos"] = np.arange(len(df))

    MM_TO_IN = 1 / 25.4
    fig_w_in = 73.5 * MM_TO_IN
    fig_h_in = 95.2 * MM_TO_IN

    AXIS_LINEWIDTH = 0.35  # thinner spines/ticks throughout

    fig, (ax_dot, ax_ridge) = plt.subplots(
        1, 2, figsize=(fig_w_in, fig_h_in), sharey=True,
        gridspec_kw={"width_ratios": [0.75, 1.25], "wspace": 0.05}
    )

    # --- Left: forest/dot plot ---
    dot_size = np.where(df["significant"], 24, 15)
    ax_dot.scatter(df["log2_enrichment"], df["y_pos"], s=dot_size, c=df["color"],
                   edgecolor="black", linewidth=0.35, zorder=3)
    ax_dot.axvline(0, color="black", linestyle="--", linewidth=AXIS_LINEWIDTH, zorder=0)
    for _, row in df[df["significant"]].iterrows():
        ax_dot.text(row["log2_enrichment"] + 0.15, row["y_pos"], "*",
                     fontsize=6, fontweight="bold", va="center")
    ax_dot.set_yticks(df["y_pos"])
    ax_dot.set_yticklabels(df["abbrev"], fontsize=6.5, fontweight="bold")
    # FIX: look up tick color using the canonical cell_type (with "+"), not raw_ct,
    # since atac_colors is keyed by canonical labels -- previously PV+/SST+/VIP+
    # silently fell back to the gray default here.
    for tick, ct in zip(ax_dot.get_yticklabels(), df["cell_type"]):
        tick.set_color(atac_colors.get(ct, "#333333"))
    ax_dot.set_xlabel("log2(Enrichment)", fontsize=6)
    ax_dot.tick_params(axis="x", labelsize=4.5, width=AXIS_LINEWIDTH, length=2)
    ax_dot.tick_params(axis="y", length=2, pad=1, width=AXIS_LINEWIDTH)
    ax_dot.grid(axis="x", linestyle=":", linewidth=0.25, alpha=0.4)
    ax_dot.spines[["top", "right"]].set_visible(False)
    for side in ["left", "bottom"]:
        ax_dot.spines[side].set_linewidth(AXIS_LINEWIDTH)
    ax_dot.set_ylim(-0.8, len(df) - 0.2)

    # --- Right: ridgeline of null distributions, centered on y0 ---
    all_null_vals = np.concatenate([null_dist_cache[ct] for ct in df["raw_ct"] if ct in null_dist_cache])
    x_grid = np.linspace(all_null_vals.min() - 1, all_null_vals.max() + 1, 300)

    baseline_offset = ridge_height / 2  # shifts ridge so it straddles y0 instead of sitting above it

    for _, row in df.iterrows():
        raw_ct = row["raw_ct"]
        if raw_ct not in null_dist_cache:
            continue
        null_vals = null_dist_cache[raw_ct]
        observed = observed_cache[raw_ct]
        y0 = row["y_pos"] - baseline_offset
        color = row["color"]

        kde = gaussian_kde(null_vals)
        density = kde(x_grid)
        density_norm = density / density.max() * ridge_height

        ax_ridge.fill_between(x_grid, y0, y0 + density_norm, color=color, alpha=0.45,
                               linewidth=0.25, edgecolor=color, zorder=2)
        ax_ridge.plot([observed, observed], [y0, y0 + ridge_height * 0.9],
                      color="black", linestyle="--", linewidth=0.4, zorder=3)

    ax_ridge.set_xlabel("Null overlap count", fontsize=6)
    ax_ridge.tick_params(axis="x", labelsize=4.5, width=AXIS_LINEWIDTH, length=2)
    ax_ridge.tick_params(axis="y", left=False, labelleft=False)
    ax_ridge.spines[["top", "right", "left"]].set_visible(False)
    ax_ridge.spines["bottom"].set_linewidth(AXIS_LINEWIDTH)
    ax_ridge.grid(axis="x", linestyle=":", linewidth=0.25, alpha=0.3)

    fig.subplots_adjust(left=0.22, right=0.98, top=0.98, bottom=0.09)

    fig.savefig(out_pdf, dpi=600)
    fig.savefig(out_png, dpi=600)
    plt.close(fig)
    print(f"Saved: {out_pdf}")
    print(f"Saved: {out_png}")


FIG_RIDGE_PDF = os.path.join(OUT_DIR, "Fig_forest_with_ridgelines_compact.pdf")
FIG_RIDGE_PNG = FIG_RIDGE_PDF.replace(".pdf", ".png")

plot_forest_with_ridgelines(
    overlap_results, null_dist_cache, observed_cache,
    abbrev_map, atac_colors, FIG_RIDGE_PDF, FIG_RIDGE_PNG
)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_forest_with_ridgelines_compact.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_forest_with_ridgelines_compact.png


In [28]:
class_colors = {"Promoter": "#1b9e77", "Enhancer": "#d95f02", "Unclassified": "#999999"}
cell_types_present = top_candidates["cell_type"].unique()

fig, axes = plt.subplots(1, len(cell_types_present), figsize=(3.2 * len(cell_types_present), 6), sharey=False)
if len(cell_types_present) == 1:
    axes = [axes]

for ax, ct in zip(axes, cell_types_present):
    sub = top_candidates[top_candidates["cell_type"] == ct].sort_values("effect_size")
    colors = sub["region_class"].map(class_colors).fillna("#999999")
    ax.barh(range(len(sub)), sub["effect_size"], color=colors, edgecolor="none")
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub["region"], fontsize=5)
    ax.axvline(0, color="black", linewidth=0.6)
    ct_color = atac_colors.get(ct, "#4d4d4d")
    n_ko = sub["n_KO_metacells"].iloc[0]
    n_ctrl = sub["n_Ctrl_metacells"].iloc[0]
    ax.set_title(f"{ct}\n(KO n={n_ko}, Ctrl n={n_ctrl})", fontsize=8, color=ct_color)
    ax.set_xlabel("Effect size\n(rank-biserial, KO vs Ctrl)", fontsize=7)
    ax.set_xlim(-1.05, 1.05)

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=l) for l, c in class_colors.items() if l != "Unclassified"]
fig.legend(handles=legend_handles, loc="upper center", ncol=2, fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, 1.05))

fig.suptitle("Top candidate differentially accessible regions per cell type\n"
             "(Exploratory — no formal significance testing; n=1 pooled library per genotype)",
             fontsize=9, y=1.1)
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig4_top_candidate_regions_exploratory.pdf")
fig.savefig(outpath, bbox_inches="tight", dpi=300)
plt.close(fig)
print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4_top_candidate_regions_exploratory.pdf


In [29]:
region_to_gene_nearest = (
    r2g.assign(abs_dist=r2g["dist_to_TSS"].abs())
       .sort_values("abs_dist")
       .drop_duplicates("Region")
       .set_index("Region")[[r2g.columns[r2g.columns.str.contains("target|Gene", case=False)][0], "dist_to_TSS", "region_class"]]
)
region_to_gene_nearest.columns = ["gene", "dist_to_TSS", "region_class_check"]

top_candidates_genes = top_candidates.merge(
    region_to_gene_nearest[["gene", "dist_to_TSS"]],
    left_on="region", right_index=True, how="left"
)

n_mapped = top_candidates_genes["gene"].notna().sum()
print(f"Mapped {n_mapped} / {top_candidates_genes.shape[0]} top candidate regions to a nearest gene")

top_candidates_genes["gene"] = top_candidates_genes["gene"].fillna("Unannotated")
top_candidates_genes["label"] = top_candidates_genes.apply(
    lambda r: f"{r['gene']} ({r['dist_to_TSS']:+.0f}bp)" if r["gene"] != "Unannotated" else r["region"],
    axis=1
)

top_candidates_genes.to_csv(os.path.join(FIG_OUT, "differential_accessibility_TOP_candidates_genes.csv"), index=False)
print(top_candidates_genes[["cell_type", "gene", "region_class", "effect_size", "log2FC"]].head(15))

Mapped 329 / 330 top candidate regions to a nearest gene
     cell_type      gene region_class  effect_size    log2FC
0   Astrocytes      Dpp8     Promoter     0.928078  0.115328
1   Astrocytes    Rangrf     Enhancer     0.919448  0.129757
2   Astrocytes    Cfap70     Enhancer    -0.918297 -0.657972
3   Astrocytes    Iqgap2     Enhancer    -0.912543 -0.864568
4   Astrocytes  Nfatc2ip     Promoter     0.912543  0.104569
5   Astrocytes    Vstm2l     Enhancer    -0.910242 -0.499791
6   Astrocytes    Spaca3     Enhancer    -0.898734 -1.170177
7   Astrocytes   Gm17597     Promoter     0.898734  0.166618
8   Astrocytes    Zfp384     Enhancer    -0.894707 -0.609082
9   Astrocytes      Tcf3     Enhancer     0.892405  0.125737
10  Astrocytes    Ppfia3     Enhancer     0.892405  0.103871
11  Astrocytes     Emc10     Enhancer     0.890104  0.129626
12  Astrocytes      Ei24     Promoter    -0.889528 -0.192694
13  Astrocytes     Myo5c     Enhancer    -0.888953 -0.455189
14  Astrocytes   Zfp518b    

In [30]:
class_colors = {"Promoter": "#1b9e77", "Enhancer": "#d95f02", "Unclassified": "#999999"}
cell_types_present = top_candidates_genes["cell_type"].unique()

fig, axes = plt.subplots(1, len(cell_types_present), figsize=(3.4 * len(cell_types_present), 6.5), sharey=False)
if len(cell_types_present) == 1:
    axes = [axes]

for ax, ct in zip(axes, cell_types_present):
    sub = top_candidates_genes[top_candidates_genes["cell_type"] == ct].sort_values("effect_size")
    colors = sub["region_class"].map(class_colors).fillna("#999999")
    ax.barh(range(len(sub)), sub["effect_size"], color=colors, edgecolor="none")
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub["label"], fontsize=5.5)
    ax.axvline(0, color="black", linewidth=0.6)
    ct_color = atac_colors.get(ct, "#4d4d4d")
    n_ko = sub["n_KO_metacells"].iloc[0]
    n_ctrl = sub["n_Ctrl_metacells"].iloc[0]
    ax.set_title(f"{ct}\n(KO n={n_ko}, Ctrl n={n_ctrl})", fontsize=8, color=ct_color)
    ax.set_xlabel("Effect size\n(rank-biserial, KO vs Ctrl)", fontsize=7)
    ax.set_xlim(-1.05, 1.05)

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=l) for l, c in class_colors.items() if l != "Unclassified"]
fig.legend(handles=legend_handles, loc="upper center", ncol=2, fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, 1.05))

fig.suptitle("Top candidate differentially accessible regions, mapped to nearest gene\n"
             "(Exploratory — no formal significance testing; n=1 pooled library per genotype)",
             fontsize=9, y=1.08)
plt.tight_layout()
outpath = os.path.join(FIG_OUT, "Fig4_top_candidate_regions_genes_exploratory.pdf")
fig.savefig(outpath, bbox_inches="tight", dpi=300)
plt.close(fig)
print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4_top_candidate_regions_genes_exploratory.pdf


In [17]:
# # In[C]: Unbiased cross-modal ATAC-RNA direction test (concordant vs discordant)
# #
# # Removes all pre-selection bias:
# # - Uses ALL bootstrap-stable ATAC-linked genes per cell type (no top-N cutoff).
# # - Tests association between ATAC direction (open/closed) and RNA direction
# #   (up/down) among genes that are BOTH stable-ATAC-linked AND BH-FDR DEGs.
# # - Reports concordant AND discordant counts from the same 2x2 table.
# # - Uses a label-permutation null (shuffling RNA direction) rather than any
# #   effect-size-based gene selection.

# import os
# import numpy as np
# import pandas as pd
# from scipy.stats import fisher_exact
# from statsmodels.stats.contingency_tables import Table2x2
# from statsmodels.stats.multitest import multipletests

# RANDOM_SEED = 42
# N_PERMUTATIONS = 10000
# MIN_GENES_PER_TEST = 5

# rng = np.random.default_rng(RANDOM_SEED)

# # gene_atac and deg_df must already exist from the earlier cells.
# # gene_atac: cell_type, gene_key, effect_direction (KO_open/KO_closed), stable_direction
# # deg_df: cell_type, gene_key, rna_log2FC, is_DEG_BH_FDR

# stable_genes = gene_atac[gene_atac["stable_direction"]].copy()

# results = []
# gene_level_tables = []

# shared_cts = sorted(
#     set(stable_genes["cell_type"]).intersection(set(deg_df["cell_type"]))
# )

# for ct in shared_cts:
#     atac_sub = stable_genes[stable_genes["cell_type"] == ct].copy()
#     deg_sub = deg_df[
#         (deg_df["cell_type"] == ct) & (deg_df["is_DEG_BH_FDR"])
#     ].copy()

#     merged = atac_sub.merge(
#         deg_sub[["gene_key", "gene", "rna_log2FC"]],
#         on="gene_key", how="inner"
#     )

#     if merged.shape[0] < MIN_GENES_PER_TEST:
#         print(f"Skipping {ct}: only {merged.shape[0]} stable ATAC-linked DEGs.")
#         continue

#     merged["rna_direction"] = np.where(merged["rna_log2FC"] > 0, "up", "down")

#     a = int(((merged["effect_direction"] == "KO_open") & (merged["rna_direction"] == "up")).sum())
#     b = int(((merged["effect_direction"] == "KO_open") & (merged["rna_direction"] == "down")).sum())
#     c = int(((merged["effect_direction"] == "KO_closed") & (merged["rna_direction"] == "up")).sum())
#     d = int(((merged["effect_direction"] == "KO_closed") & (merged["rna_direction"] == "down")).sum())

#     table = np.array([[a, b], [c, d]])

#     n_concordant = a + d
#     n_discordant = b + c
#     n_total = n_concordant + n_discordant

#     _, fisher_p = fisher_exact(table, alternative="two-sided")

#     ci_table = Table2x2(table.astype(float) + 0.5)
#     odds_ratio = ci_table.oddsratio
#     ci_low, ci_high = ci_table.oddsratio_confint(alpha=0.05)

#     # Permutation null: shuffle RNA direction labels within this cell type's
#     # stable-ATAC-linked DEG set, preserving the marginal up/down proportion.
#     rna_dir_arr = merged["rna_direction"].to_numpy()
#     atac_dir_arr = (merged["effect_direction"] == "KO_open").to_numpy()
#     n_up_observed = int((rna_dir_arr == "up").sum())
#     n_genes = len(merged)

#     perm_concordant = np.empty(N_PERMUTATIONS, dtype=int)
#     for i in range(N_PERMUTATIONS):
#         perm_up_idx = rng.choice(n_genes, size=n_up_observed, replace=False)
#         perm_is_up = np.zeros(n_genes, dtype=bool)
#         perm_is_up[perm_up_idx] = True
#         perm_concordant[i] = int(
#             ((atac_dir_arr & perm_is_up) | (~atac_dir_arr & ~perm_is_up)).sum()
#         )

#     empirical_p = (np.sum(perm_concordant >= n_concordant) + 1) / (N_PERMUTATIONS + 1)
#     expected_concordant = perm_concordant.mean()

#     results.append({
#         "cell_type": ct,
#         "n_stable_ATAC_linked_DEGs": n_total,
#         "n_KO_open_RNA_up": a,
#         "n_KO_open_RNA_down": b,
#         "n_KO_closed_RNA_up": c,
#         "n_KO_closed_RNA_down": d,
#         "n_concordant": n_concordant,
#         "n_discordant": n_discordant,
#         "concordant_fraction": n_concordant / n_total,
#         "odds_ratio_descriptive": odds_ratio,
#         "odds_ratio_95CI_low": ci_low,
#         "odds_ratio_95CI_high": ci_high,
#         "fisher_p_two_sided": fisher_p,
#         "empirical_permutation_p": empirical_p,
#         "expected_concordant_permutation": expected_concordant,
#         "observed_minus_expected_concordant": n_concordant - expected_concordant,
#     })

#     merged["cell_type"] = ct
#     gene_level_tables.append(merged)

# results_df = pd.DataFrame(results)

# if results_df.empty:
#     raise ValueError("No cell types had enough stable ATAC-linked DEGs to test.")

# results_df["permutation_FDR_BH"] = multipletests(
#     results_df["empirical_permutation_p"].fillna(1.0), method="fdr_bh"
# )[1]
# results_df["fisher_FDR_BH"] = multipletests(
#     results_df["fisher_p_two_sided"].fillna(1.0), method="fdr_bh"
# )[1]

# results_df = results_df.sort_values("permutation_FDR_BH").reset_index(drop=True)

# gene_level_df = pd.concat(gene_level_tables, ignore_index=True) if gene_level_tables else pd.DataFrame()

# results_path = os.path.join(CROSSMODAL_OUT, "ATAC_RNA_unbiased_concordance_discordance.csv")
# gene_path = os.path.join(CROSSMODAL_OUT, "ATAC_RNA_unbiased_gene_level_table.csv")

# results_df.to_csv(results_path, index=False)
# gene_level_df.to_csv(gene_path, index=False)

# print("\n=== Unbiased concordant vs discordant results (all stable genes, no top-N) ===")
# print(results_df[[
#     "cell_type", "n_stable_ATAC_linked_DEGs", "n_concordant", "n_discordant",
#     "odds_ratio_descriptive", "empirical_permutation_p", "permutation_FDR_BH"
# ]].to_string(index=False))

# print(f"\nSaved: {results_path}")
# print(f"Saved: {gene_path}")

Skipping Meningeal fibroblasts: only 0 stable ATAC-linked DEGs.

=== Unbiased concordant vs discordant results (all stable genes, no top-N) ===
                            cell_type  n_stable_ATAC_linked_DEGs  n_concordant  n_discordant  odds_ratio_descriptive  empirical_permutation_p  permutation_FDR_BH
                           Astrocytes                       1410           714           696                1.004336                 0.508449            0.598547
Deep-layer extratelencephalic neurons                       2012           992          1020                1.005398                 0.499350            0.598547
                    Endothelial cells                         69            34            35                1.638806                 0.411559            0.598547
                 Layer 2/3 IT neurons                       1366           699           667                1.020537                 0.457154            0.598547
              Layer 4 sensory neurons         

In [18]:
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.lines as mlines

# # =============================================================================
# # Your fixed cell-type color palette (from atac_colors)
# # =============================================================================

# celltype_order = [
#     "Layer 2/3 IT neurons", "Layer 4 sensory neurons", "Layer 5a IT neurons",
#     "Layer 5b PT neurons", "Layer 5/6 IT neurons", "Layer 6a corticothalamic neurons",
#     "Layer 6b neurons", "Deep-layer extratelencephalic neurons",
#     "Corticospinal neurons Type I", "Corticospinal neurons Type II",
#     "PV+ interneurons", "SST+ interneurons", "VIP+ interneurons",
#     "Astrocytes", "Oligodendrocyte precursor cells", "Oligodendrocytes",
#     "Perineuronal oligodendrocytes", "Myelinating oligodendrocytes",
#     "Microglia", "Endothelial cells", "Leptomeningeal cells", "Meningeal fibroblasts"
# ]

# hex_colors = [
#     "6B5B95", "45B8AC", "955251", "4E84C4", "B565A7", "88B04B", "7B6888",
#     "C3447A", "009B77", "EFC050", "7FCDCD", "DD4124", "5B5EA6", "E07A5F",
#     "4BACC6", "E8A0BF", "ea8a33", "9B2335", "C17BAE", "DECF3F", "789262", "BC243C"
# ]

# atac_colors = {ct: f"#{h}" for ct, h in zip(celltype_order, hex_colors)}

# # =============================================================================
# # Load unbiased concordant/discordant results (from Cell C)
# # =============================================================================

# results_path = os.path.join(CROSSMODAL_OUT, "ATAC_RNA_unbiased_concordance_discordance.csv")
# plot_df = pd.read_csv(results_path)

# plot_df["color"] = plot_df["cell_type"].map(atac_colors).fillna("#999999")
# plot_df["significant"] = plot_df["permutation_FDR_BH"] < 0.05
# plot_df["log2_OR"] = np.log2(plot_df["odds_ratio_descriptive"].clip(lower=0.02))
# plot_df["log2_OR_CI_low"] = np.log2(plot_df["odds_ratio_95CI_low"].clip(lower=0.02))
# plot_df["log2_OR_CI_high"] = np.log2(plot_df["odds_ratio_95CI_high"].clip(lower=0.02))

# plot_df = plot_df.sort_values("log2_OR", ascending=True).reset_index(drop=True)
# plot_df["y_pos"] = np.arange(len(plot_df))

# # =============================================================================
# # Figure sized to fit a W24 x H63 cm panel
# # =============================================================================

# fig_w_in = 24 / 2.54
# fig_h_in = 63 / 2.54

# fig, (ax_or, ax_counts) = plt.subplots(
#     1, 2, figsize=(fig_w_in, fig_h_in),
#     gridspec_kw={"width_ratios": [1.35, 1]}
# )

# # --- Left panel: log2 odds ratio forest plot, colored by cell-type cluster ---

# ax_or.errorbar(
#     x=plot_df["log2_OR"], y=plot_df["y_pos"],
#     xerr=[
#         plot_df["log2_OR"] - plot_df["log2_OR_CI_low"],
#         plot_df["log2_OR_CI_high"] - plot_df["log2_OR"]
#     ],
#     fmt="none", ecolor="#777777", elinewidth=1.4, capsize=3, zorder=1
# )

# ax_or.scatter(
#     plot_df["log2_OR"], plot_df["y_pos"],
#     s=180, c=plot_df["color"], edgecolor="black", linewidth=0.8, zorder=3
# )

# ax_or.axvline(0, color="black", linestyle="--", linewidth=1.2, zorder=0)

# for _, row in plot_df[plot_df["significant"]].iterrows():
#     ax_or.text(
#         row["log2_OR"] + 0.25, row["y_pos"], "*",
#         fontsize=18, fontweight="bold", color="black", va="center"
#     )

# ax_or.set_yticks(plot_df["y_pos"])
# ax_or.set_yticklabels(plot_df["cell_type"], fontsize=11)
# ax_or.set_xlabel("log2(Odds ratio)\nConcordant vs discordant RNA direction", fontsize=11)
# ax_or.set_title("ATAC-RNA directional concordance", fontsize=13, fontweight="bold")
# ax_or.grid(axis="x", linestyle=":", linewidth=0.6, alpha=0.5)
# ax_or.spines[["top", "right"]].set_visible(False)

# for tick, ct in zip(ax_or.get_yticklabels(), plot_df["cell_type"]):
#     tick.set_color(atac_colors.get(ct, "#333333"))

# # --- Right panel: stacked concordant/discordant gene counts ---

# ax_counts.barh(
#     plot_df["y_pos"], plot_df["n_concordant"],
#     color=plot_df["color"], edgecolor="black", linewidth=0.5,
#     label="Concordant", height=0.7, zorder=2
# )
# ax_counts.barh(
#     plot_df["y_pos"], plot_df["n_discordant"],
#     left=plot_df["n_concordant"],
#     color=plot_df["color"], edgecolor="black", linewidth=0.5,
#     alpha=0.35, height=0.7, zorder=2
# )

# for _, row in plot_df.iterrows():
#     ax_counts.text(
#         row["n_concordant"] + row["n_discordant"] + 0.6, row["y_pos"],
#         f"n={int(row['n_stable_ATAC_linked_DEGs'])}",
#         va="center", fontsize=9, color="#333333"
#     )

# ax_counts.set_yticks(plot_df["y_pos"])
# ax_counts.set_yticklabels([])
# ax_counts.set_xlabel("Stable ATAC-linked DEGs\n(solid = concordant, faded = discordant)", fontsize=11)
# ax_counts.set_title("Gene counts by direction", fontsize=13, fontweight="bold")
# ax_counts.spines[["top", "right"]].set_visible(False)
# ax_counts.grid(axis="x", linestyle=":", linewidth=0.6, alpha=0.5)

# fig.suptitle(
#     "Cross-modal ATAC-RNA concordance by cell type\n"
#     "Unbiased test: all bootstrap-stable regions, no effect-size pre-selection",
#     fontsize=14, fontweight="bold", y=0.995
# )

# legend_handles = [
#     mlines.Line2D([], [], marker="*", color="black", linestyle="None",
#                   markersize=12, label="FDR < 0.05")
# ]
# fig.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(0.98, 0.97), fontsize=10, frameon=False)

# plt.tight_layout(rect=[0, 0, 1, 0.97])

# out_path = os.path.join(CROSSMODAL_OUT, "Fig_ATAC_RNA_unbiased_concordance_forestplot.pdf")
# png_path = out_path.replace(".pdf", ".png")

# fig.savefig(out_path, bbox_inches="tight", dpi=300)
# fig.savefig(png_path, bbox_inches="tight", dpi=300)
# plt.close(fig)

# print(f"Saved: {out_path}")
# print(f"Saved: {png_path}")

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_ATAC_RNA_unbiased_concordance_forestplot.pdf
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/cross_modal_ATAC_RNA_concordance/Fig_ATAC_RNA_unbiased_concordance_forestplot.png


In [20]:
# # In[D]: eRegulon-guided ATAC-RNA concordance test
# #
# # Uses the TF x cell-type SCENIC+ eRegulon matrices (region-based ATAC AUC delta,
# # gene-based AUC delta, TF expression delta), which were derived from network
# # topology (TF motif/target inference), NOT from KO-vs-Ctrl effect size.
# # This makes the TF list a legitimate pre-specified candidate set: no selection
# # on the outcome we're about to test.

# import numpy as np
# import pandas as pd
# from scipy.stats import fisher_exact, pearsonr
# from statsmodels.stats.contingency_tables import Table2x2
# from statsmodels.stats.multitest import multipletests

# RANDOM_SEED = 42
# N_PERMUTATIONS = 10000
# MIN_TFS_PER_TEST = 5
# DIRECTION_EPS = 1e-6  # values within +/- this are treated as "no change"

# rng = np.random.default_rng(RANDOM_SEED)

# region_delta = pd.read_csv("ereg_delta_norm_region_AUC_Neuronal_and_Glial_Only.csv", index_col=0)
# gene_delta   = pd.read_csv("ereg_delta_norm_gene_AUC_Neuronal_and_Glial_Only.csv", index_col=0)
# tf_delta     = pd.read_csv("ereg_delta_norm_TF_expr_Neuronal_and_Glial_Only.csv", index_col=0)

# common_tfs = region_delta.index.intersection(gene_delta.index).intersection(tf_delta.index)
# common_cts = region_delta.columns.intersection(gene_delta.columns).intersection(tf_delta.columns)

# region_delta = region_delta.loc[common_tfs, common_cts]
# gene_delta   = gene_delta.loc[common_tfs, common_cts]
# tf_delta     = tf_delta.loc[common_tfs, common_cts]

# def sign_direction(x):
#     return np.where(x > DIRECTION_EPS, "up", np.where(x < -DIRECTION_EPS, "down", "flat"))

# results = []
# scatter_rows = []

# for ct in common_cts:
#     atac_vals = region_delta[ct].to_numpy()
#     rna_vals  = gene_delta[ct].to_numpy()

#     valid = (sign_direction(atac_vals) != "flat") & (sign_direction(rna_vals) != "flat")
#     if valid.sum() < MIN_TFS_PER_TEST:
#         continue

#     atac_sub = atac_vals[valid]
#     rna_sub  = rna_vals[valid]
#     tf_names = common_tfs[valid]

#     atac_dir = sign_direction(atac_sub)
#     rna_dir  = sign_direction(rna_sub)

#     a = int(((atac_dir == "up") & (rna_dir == "up")).sum())
#     b = int(((atac_dir == "up") & (rna_dir == "down")).sum())
#     c = int(((atac_dir == "down") & (rna_dir == "up")).sum())
#     d = int(((atac_dir == "down") & (rna_dir == "down")).sum())

#     table = np.array([[a, b], [c, d]])
#     n_concordant = a + d
#     n_discordant = b + c
#     n_total = n_concordant + n_discordant

#     _, fisher_p = fisher_exact(table, alternative="two-sided")
#     ci_table = Table2x2(table.astype(float) + 0.5)
#     odds_ratio = ci_table.oddsratio
#     ci_low, ci_high = ci_table.oddsratio_confint(alpha=0.05)

#     r_pearson, p_pearson = pearsonr(atac_sub, rna_sub)

#     is_up = (atac_dir == "up")
#     n_up_observed = int((rna_dir == "up").sum())
#     n_tfs = len(tf_names)

#     perm_concordant = np.empty(N_PERMUTATIONS, dtype=int)
#     for i in range(N_PERMUTATIONS):
#         perm_up_idx = rng.choice(n_tfs, size=n_up_observed, replace=False)
#         perm_is_up = np.zeros(n_tfs, dtype=bool)
#         perm_is_up[perm_up_idx] = True
#         perm_concordant[i] = int(((is_up & perm_is_up) | (~is_up & ~perm_is_up)).sum())

#     empirical_p = (np.sum(perm_concordant >= n_concordant) + 1) / (N_PERMUTATIONS + 1)
#     expected_concordant = perm_concordant.mean()

#     results.append({
#         "cell_type": ct,
#         "n_TFs_tested": n_total,
#         "n_ATAC_up_RNA_up": a,
#         "n_ATAC_up_RNA_down": b,
#         "n_ATAC_down_RNA_up": c,
#         "n_ATAC_down_RNA_down": d,
#         "n_concordant": n_concordant,
#         "n_discordant": n_discordant,
#         "concordant_fraction": n_concordant / n_total,
#         "odds_ratio_descriptive": odds_ratio,
#         "odds_ratio_95CI_low": ci_low,
#         "odds_ratio_95CI_high": ci_high,
#         "fisher_p_two_sided": fisher_p,
#         "pearson_r_ATAC_vs_RNA_delta": r_pearson,
#         "pearson_p": p_pearson,
#         "empirical_permutation_p": empirical_p,
#         "expected_concordant_permutation": expected_concordant,
#     })

#     for tf, av, rv in zip(tf_names, atac_sub, rna_sub):
#         scatter_rows.append({"cell_type": ct, "TF": tf, "ATAC_delta": av, "RNA_delta": rv})

# results_df = pd.DataFrame(results)
# results_df["permutation_FDR_BH"] = multipletests(
#     results_df["empirical_permutation_p"].fillna(1.0), method="fdr_bh"
# )[1]
# results_df["fisher_FDR_BH"] = multipletests(
#     results_df["fisher_p_two_sided"].fillna(1.0), method="fdr_bh"
# )[1]
# results_df["pearson_FDR_BH"] = multipletests(
#     results_df["pearson_p"].fillna(1.0), method="fdr_bh"
# )[1]

# results_df = results_df.sort_values("permutation_FDR_BH").reset_index(drop=True)
# scatter_df = pd.DataFrame(scatter_rows)

# results_df.to_csv("eRegulon_guided_ATAC_RNA_concordance.csv", index=False)
# scatter_df.to_csv("eRegulon_guided_ATAC_RNA_TF_level_deltas.csv", index=False)

# print(results_df[[
#     "cell_type", "n_TFs_tested", "n_concordant", "n_discordant",
#     "odds_ratio_descriptive", "pearson_r_ATAC_vs_RNA_delta", "pearson_p",
#     "empirical_permutation_p", "permutation_FDR_BH"
# ]].to_string(index=False))

                            cell_type  n_TFs_tested  n_concordant  n_discordant  odds_ratio_descriptive  pearson_r_ATAC_vs_RNA_delta  pearson_p  empirical_permutation_p  permutation_FDR_BH
                           Astrocytes            59            39            20                3.000000                     0.156920   0.235263                 0.041996            0.314969
                 Layer 2/3 IT neurons            59            38            21                3.168421                     0.161391   0.222022                 0.027897            0.314969
      Oligodendrocyte precursor cells            59            22            37                0.385180                    -0.038511   0.772133                 0.981602            0.995100
                            Microglia            59            24            35                0.540373                     0.059302   0.655488                 0.925607            0.995100
                     Oligodendrocytes            59    

In [21]:
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.lines as mlines
# from scipy.stats import pearsonr

# # =============================================================================
# # Your fixed cell-type color palette
# # =============================================================================

# celltype_order = [
#     "Layer 2/3 IT neurons", "Layer 4 sensory neurons", "Layer 5a IT neurons",
#     "Layer 5b PT neurons", "Layer 5/6 IT neurons", "Layer 6a corticothalamic neurons",
#     "Layer 6b neurons", "Deep-layer extratelencephalic neurons",
#     "Corticospinal neurons Type I", "Corticospinal neurons Type II",
#     "PV+ interneurons", "SST+ interneurons", "VIP+ interneurons",
#     "Astrocytes", "Oligodendrocyte precursor cells", "Oligodendrocytes",
#     "Perineuronal oligodendrocytes", "Myelinating oligodendrocytes",
#     "Microglia", "Endothelial cells", "Leptomeningeal cells", "Meningeal fibroblasts"
# ]
# hex_colors = [
#     "6B5B95", "45B8AC", "955251", "4E84C4", "B565A7", "88B04B", "7B6888",
#     "C3447A", "009B77", "EFC050", "7FCDCD", "DD4124", "5B5EA6", "E07A5F",
#     "4BACC6", "E8A0BF", "ea8a33", "9B2335", "C17BAE", "DECF3F", "789262", "BC243C"
# ]
# atac_colors = {ct: f"#{h}" for ct, h in zip(celltype_order, hex_colors)}

# # =============================================================================
# # Load results
# # =============================================================================

# results_df = pd.read_csv("eRegulon_guided_ATAC_RNA_concordance.csv")
# scatter_df = pd.read_csv("eRegulon_guided_ATAC_RNA_TF_level_deltas.csv")

# results_df["color"] = results_df["cell_type"].map(atac_colors).fillna("#999999")
# results_df = results_df.sort_values("pearson_r_ATAC_vs_RNA_delta", ascending=True).reset_index(drop=True)
# results_df["y_pos"] = np.arange(len(results_df))
# results_df["significant_nominal"] = results_df["pearson_p"] < 0.05
# results_df["significant_FDR"] = results_df["permutation_FDR_BH"] < 0.05

# # =============================================================================
# # Figure: left = correlation summary forest, right = TF scatter grid (top 4 cts)
# # =============================================================================

# fig = plt.figure(figsize=(24 / 2.54, 40 / 2.54))
# gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.3], hspace=0.35)

# # --- Top: Pearson r summary, all cell types, colored by cluster ---
# ax_r = fig.add_subplot(gs[0])

# ax_r.errorbar(
#     results_df["pearson_r_ATAC_vs_RNA_delta"], results_df["y_pos"],
#     fmt="none", ecolor="#bbbbbb", zorder=1
# )
# ax_r.scatter(
#     results_df["pearson_r_ATAC_vs_RNA_delta"], results_df["y_pos"],
#     s=160, c=results_df["color"], edgecolor="black", linewidth=0.8, zorder=3
# )
# ax_r.axvline(0, color="black", linestyle="--", linewidth=1.1, zorder=0)

# for _, row in results_df[results_df["significant_nominal"]].iterrows():
#     marker = "**" if row["significant_FDR"] else "*"
#     ax_r.text(row["pearson_r_ATAC_vs_RNA_delta"] + 0.03, row["y_pos"], marker,
#                fontsize=13, fontweight="bold", color="black", va="center")

# ax_r.set_yticks(results_df["y_pos"])
# ax_r.set_yticklabels(results_df["cell_type"], fontsize=10)
# for tick, ct in zip(ax_r.get_yticklabels(), results_df["cell_type"]):
#     tick.set_color(atac_colors.get(ct, "#333333"))

# ax_r.set_xlabel("Pearson r (ATAC delta vs RNA delta, eRegulon TFs)", fontsize=11)
# ax_r.set_title(
#     "eRegulon-guided ATAC-RNA concordance by cell type\n"
#     "Pre-specified TF set (SCENIC+ network), * nominal p<0.05, ** FDR<0.05",
#     fontsize=12, fontweight="bold"
# )
# ax_r.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.5)
# ax_r.spines[["top", "right"]].set_visible(False)

# # --- Bottom: TF-level scatter grid for the top 4 |r| cell types ---
# top_cts = (
#     results_df.reindex(results_df["pearson_r_ATAC_vs_RNA_delta"].abs().sort_values(ascending=False).index)
#     ["cell_type"].head(4).tolist()
# )

# gs_bottom = gs[1].subgridspec(2, 2, wspace=0.3, hspace=0.4)

# for i, ct in enumerate(top_cts):
#     ax = fig.add_subplot(gs_bottom[i // 2, i % 2])
#     sub = scatter_df[scatter_df["cell_type"] == ct]
#     color = atac_colors.get(ct, "#999999")

#     ax.scatter(sub["ATAC_delta"], sub["RNA_delta"], s=45, color=color,
#                edgecolor="black", linewidth=0.4, alpha=0.85, zorder=3)

#     if len(sub) > 2:
#         m, b = np.polyfit(sub["ATAC_delta"], sub["RNA_delta"], 1)
#         xs = np.linspace(sub["ATAC_delta"].min(), sub["ATAC_delta"].max(), 100)
#         ax.plot(xs, m * xs + b, color=color, linewidth=2, zorder=2)

#     r, p = pearsonr(sub["ATAC_delta"], sub["RNA_delta"])
#     ax.axhline(0, color="#cccccc", linewidth=0.8, zorder=1)
#     ax.axvline(0, color="#cccccc", linewidth=0.8, zorder=1)
#     ax.set_title(f"{ct}\nr={r:.2f}, p={p:.3f}, n={len(sub)}", fontsize=9.5, color=color)
#     ax.set_xlabel("ATAC eRegulon delta (KO-Ctrl)", fontsize=8.5)
#     ax.set_ylabel("RNA eRegulon delta (KO-Ctrl)", fontsize=8.5)
#     ax.tick_params(labelsize=8)
#     ax.spines[["top", "right"]].set_visible(False)

# fig.suptitle(
#     "TF-level ATAC vs RNA eRegulon activity shifts (KO vs Ctrl)\n"
#     "Top 4 cell types by absolute correlation strength",
#     fontsize=13, fontweight="bold", y=1.0
# )

# plt.tight_layout(rect=[0, 0, 1, 0.97])

# os.makedirs("output", exist_ok=True)
# out_path = "output/Fig_eRegulon_guided_ATAC_RNA_concordance.pdf"
# png_path = out_path.replace(".pdf", ".png")
# fig.savefig(out_path, bbox_inches="tight", dpi=300)
# fig.savefig(png_path, bbox_inches="tight", dpi=300)
# plt.close(fig)

# print(f"Saved: {out_path}")
# print(f"Saved: {png_path}")

/var/folders/c5/9gnt7c7x7x59ygrp90w3n60r0000gn/T/ipykernel_38693/3438344803.py:117: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.97])


Saved: output/Fig_eRegulon_guided_ATAC_RNA_concordance.pdf
Saved: output/Fig_eRegulon_guided_ATAC_RNA_concordance.png


In [ ]:
###############################

In [22]:
# results = []
# for ct in meta["cell_type"].unique():
#     ct_mask = meta["cell_type"] == ct
#     ko_cells = meta.index[ct_mask & (meta["genotype"] == "KO")]
#     ctrl_cells = meta.index[ct_mask & (meta["genotype"] == "Ctrl")]
#     if len(ko_cells) < 10 or len(ctrl_cells) < 10:
#         continue

#     sub = acc_df.loc[list(ko_cells) + list(ctrl_cells)]
#     var_regions = sub.columns[sub.var(axis=0) > 0]
#     ko_mat = acc_df.loc[ko_cells, var_regions].values
#     ctrl_mat = acc_df.loc[ctrl_cells, var_regions].values

#     log2fc = np.log2(ko_mat.mean(axis=0) + 1e-3) - np.log2(ctrl_mat.mean(axis=0) + 1e-3)
#     pvals = np.ones(len(var_regions))
#     for i in range(len(var_regions)):
#         try:
#             _, p = mannwhitneyu(ko_mat[:, i], ctrl_mat[:, i], alternative="two-sided")
#             pvals[i] = p
#         except ValueError:
#             pvals[i] = 1.0

#     _, padj, _, _ = multipletests(pvals, method="fdr_bh")
#     res = pd.DataFrame({"region": var_regions, "log2FC": log2fc, "pval": pvals,
#                          "padj": padj, "cell_type": ct})
#     results.append(res)

# diff_acc = pd.concat(results, ignore_index=True)
# diff_acc.to_csv(os.path.join(FIG_OUT, "differential_accessibility_KOvsCtrl.csv"), index=False)

# # cell_types_present = diff_acc["cell_type"].unique()
# # fig, axes = plt.subplots(1, len(cell_types_present), figsize=(4.5 * len(cell_types_present), 4.5))
# # if len(cell_types_present) == 1:
# #     axes = [axes]
# # for ax, ct in zip(axes, cell_types_present):
# #     sub = diff_acc[diff_acc["cell_type"] == ct]
# #     sig_up = sub[(sub["padj"] < 0.05) & (sub["log2FC"] > 0.5)]
# #     sig_down = sub[(sub["padj"] < 0.05) & (sub["log2FC"] < -0.5)]
# #     ns = sub.drop(index=sig_up.index.union(sig_down.index))

# #     ct_color = atac_colors.get(ct, "#4d4d4d")
# #     ax.scatter(ns["log2FC"], -np.log10(ns["padj"] + 1e-300), s=4, color="lightgrey")
# #     ax.scatter(sig_up["log2FC"], -np.log10(sig_up["padj"] + 1e-300), s=6, color=ct_color)
# #     ax.scatter(sig_down["log2FC"], -np.log10(sig_down["padj"] + 1e-300), s=6, color=ct_color, alpha=0.5)
# #     ax.axhline(-np.log10(0.05), color="grey", linestyle="--", linewidth=0.8)
# #     ax.set_title(ct, fontsize=10, color=ct_color)
# #     ax.set_xlabel("log2FC (KO vs Ctrl)")
# #     ax.set_ylabel("-log10(padj)")
# # plt.tight_layout()
# # outpath = os.path.join(FIG_OUT, "Fig4_differential_accessibility_volcano.pdf")
# # fig.savefig(outpath, bbox_inches="tight")
# # plt.close(fig)
# # print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4_differential_accessibility_volcano.pdf


In [5]:
# MIN_MEAN_ACC = 0.05   # region must be reasonably accessible in at least one group
# MIN_VAR = 1e-6

# results = []
# for ct in meta["cell_type"].unique():
#     ct_mask = meta["cell_type"] == ct
#     ko_cells = meta.index[ct_mask & (meta["genotype"] == "KO")]
#     ctrl_cells = meta.index[ct_mask & (meta["genotype"] == "Ctrl")]
#     if len(ko_cells) < 10 or len(ctrl_cells) < 10:
#         continue

#     ko_mean_all = acc_df.loc[ko_cells].mean(axis=0)
#     ctrl_mean_all = acc_df.loc[ctrl_cells].mean(axis=0)
#     var_all = acc_df.loc[list(ko_cells) + list(ctrl_cells)].var(axis=0)

#     keep = ((ko_mean_all > MIN_MEAN_ACC) | (ctrl_mean_all > MIN_MEAN_ACC)) & (var_all > MIN_VAR)
#     var_regions = acc_df.columns[keep]
#     print(f"{ct}: testing {len(var_regions)} / {acc_df.shape[1]} regions after filtering")

#     if len(var_regions) == 0:
#         continue

#     ko_mat = acc_df.loc[ko_cells, var_regions].values
#     ctrl_mat = acc_df.loc[ctrl_cells, var_regions].values

#     log2fc = np.log2(ko_mat.mean(axis=0) + 1e-3) - np.log2(ctrl_mat.mean(axis=0) + 1e-3)
#     pvals = np.ones(len(var_regions))
#     for i in range(len(var_regions)):
#         try:
#             _, p = mannwhitneyu(ko_mat[:, i], ctrl_mat[:, i], alternative="two-sided")
#             pvals[i] = p
#         except ValueError:
#             pvals[i] = 1.0

#     _, padj, _, _ = multipletests(pvals, method="fdr_bh")
#     res = pd.DataFrame({"region": var_regions, "log2FC": log2fc, "pval": pvals,
#                          "padj": padj, "cell_type": ct})
#     results.append(res)

# diff_acc = pd.concat(results, ignore_index=True)

# # Clip extreme values so plotting axes stay sane; keep raw values in CSV separately if needed
# diff_acc["padj_clipped"] = diff_acc["padj"].clip(lower=1e-30)
# diff_acc["log2FC_clipped"] = diff_acc["log2FC"].clip(-10, 10)

# diff_acc.to_csv(os.path.join(FIG_OUT, "differential_accessibility_KOvsCtrl.csv"), index=False)
# print(f"Final table: {diff_acc.shape[0]} rows -> CSV size should now be much smaller")

# cell_types_present = diff_acc["cell_type"].unique()
# fig, axes = plt.subplots(1, len(cell_types_present), figsize=(4.5 * len(cell_types_present), 4.5))
# if len(cell_types_present) == 1:
#     axes = [axes]
# for ax, ct in zip(axes, cell_types_present):
#     sub = diff_acc[diff_acc["cell_type"] == ct]
#     sig_up = sub[(sub["padj"] < 0.05) & (sub["log2FC_clipped"] > 0.5)]
#     sig_down = sub[(sub["padj"] < 0.05) & (sub["log2FC_clipped"] < -0.5)]
#     ns = sub.drop(index=sig_up.index.union(sig_down.index))

#     ct_color = atac_colors.get(ct, "#4d4d4d")
#     # rasterized=True keeps the PDF small — points become a raster image layer, not millions of vector objects
#     ax.scatter(ns["log2FC_clipped"], -np.log10(ns["padj_clipped"]), s=3, color="lightgrey",
#                rasterized=True, alpha=0.5)
#     ax.scatter(sig_up["log2FC_clipped"], -np.log10(sig_up["padj_clipped"]), s=5, color=ct_color,
#                rasterized=True)
#     ax.scatter(sig_down["log2FC_clipped"], -np.log10(sig_down["padj_clipped"]), s=5, color=ct_color,
#                alpha=0.5, rasterized=True)
#     ax.axhline(-np.log10(0.05), color="grey", linestyle="--", linewidth=0.8)
#     ax.set_xlim(-10.5, 10.5)
#     ax.set_ylim(0, 32)
#     ax.set_title(ct, fontsize=10, color=ct_color)
#     ax.set_xlabel("log2FC (KO vs Ctrl), clipped at +/-10")
#     ax.set_ylabel("-log10(padj), clipped at 30")
# plt.tight_layout()

# outpath = os.path.join(FIG_OUT, "Fig4_differential_accessibility_volcano.pdf")
# fig.savefig(outpath, bbox_inches="tight", dpi=300)  # dpi controls rasterized-layer resolution
# plt.close(fig)
# print("Saved:", outpath)

Astrocytes: testing 352343 / 746029 regions after filtering
Deep-layer extratelencephalic neurons: testing 461561 / 746029 regions after filtering
Layer 4 sensory neurons: testing 450302 / 746029 regions after filtering
Layer 5/6 IT neurons: testing 461421 / 746029 regions after filtering
Layer 5a IT neurons: testing 449197 / 746029 regions after filtering
Layer 6a corticothalamic neurons: testing 444560 / 746029 regions after filtering
Microglia: testing 432151 / 746029 regions after filtering
Oligodendrocytes: testing 291296 / 746029 regions after filtering
PV+ interneurons: testing 468144 / 746029 regions after filtering
SST+ interneurons: testing 499986 / 746029 regions after filtering
VIP+ interneurons: testing 481985 / 746029 regions after filtering
Final table: 4792946 rows -> CSV size should now be much smaller
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4_differential_accessibility_volcano.pdf


In [9]:
# print(meta.groupby(["cell_type", "genotype"]).size().unstack())

genotype                               Ctrl   KO
cell_type                                       
Astrocytes                               22  158
Deep-layer extratelencephalic neurons    14   60
Endothelial cells                         4   22
Layer 2/3 IT neurons                      4   18
Layer 4 sensory neurons                 106  408
Layer 5/6 IT neurons                     46  200
Layer 5a IT neurons                      88  362
Layer 5b PT neurons                       8   30
Layer 6a corticothalamic neurons         50  182
Layer 6b neurons                          2   12
Meningeal fibroblasts                     8   26
Microglia                                14   52
Oligodendrocyte precursor cells           8   38
Oligodendrocytes                         42  196
PV+ interneurons                         18   78
SST+ interneurons                        18   58
VIP+ interneurons                        20   74


In [19]:
# depth_check = acc_df.sum(axis=1).to_frame("total_accessibility").join(meta)
# print(depth_check.groupby(["cell_type", "genotype"])["total_accessibility"].median().unstack())

genotype                                    Ctrl         KO
cell_type                                                  
Astrocytes                             719672.90  721157.75
Deep-layer extratelencephalic neurons  621496.65  625717.45
Endothelial cells                      610116.05  612097.65
Layer 2/3 IT neurons                   656912.70  660763.35
Layer 4 sensory neurons                649000.00  659526.15
Layer 5/6 IT neurons                   637394.70  640542.90
Layer 5a IT neurons                    640163.05  649052.75
Layer 5b PT neurons                    632041.70  641793.25
Layer 6a corticothalamic neurons       643265.85  651978.35
Layer 6b neurons                       619032.45  627591.90
Meningeal fibroblasts                  679175.70  696252.10
Microglia                              725174.55  744060.75
Oligodendrocyte precursor cells        650409.40  673027.85
Oligodendrocytes                       738988.75  748438.40
PV+ interneurons                       6

In [20]:
# from sklearn.preprocessing import quantile_transform

# acc_df_qnorm = acc_df.copy()
# for ct in meta["cell_type"].unique():
#     ct_cells = meta.index[meta["cell_type"] == ct]
#     if len(ct_cells) < 10:
#         continue
#     sub = acc_df.loc[ct_cells]
#     qn = quantile_transform(sub.values, axis=0, n_quantiles=min(1000, sub.shape[0]),
#                              output_distribution="uniform", copy=True)
#     acc_df_qnorm.loc[ct_cells] = qn

# print("Depth check after quantile normalization:")
# depth_check_qn = acc_df_qnorm.sum(axis=1).to_frame("total_accessibility").join(meta)
# print(depth_check_qn.groupby(["cell_type", "genotype"])["total_accessibility"].median().unstack())

Depth check after quantile normalization:
genotype                                        Ctrl             KO
cell_type                                                          
Astrocytes                             220842.806797  146879.280726
Deep-layer extratelencephalic neurons  233757.304794  231162.880137
Endothelial cells                      250314.190000  326740.730000
Layer 2/3 IT neurons                   230819.904762  201336.571429
Layer 4 sensory neurons                253880.364035  216365.533138
Layer 5/6 IT neurons                   247711.181633  225501.884694
Layer 5a IT neurons                    248268.917038  217092.270045
Layer 5b PT neurons                    266140.614865  232406.662162
Layer 6a corticothalamic neurons       243994.484536  219518.222173
Layer 6b neurons                       261816.903846  230668.557692
Meningeal fibroblasts                  269693.971573  219342.081890
Microglia                              279207.210322  182640.371784
Oligod

In [21]:
# row_sums = acc_df.sum(axis=1)
# target_depth = row_sums.median()
# acc_df_norm = acc_df.div(row_sums, axis=0) * target_depth

# print("Depth check after library-size normalization:")
# depth_check_norm = acc_df_norm.sum(axis=1).to_frame("total_accessibility").join(meta)
# print(depth_check_norm.groupby(["cell_type", "genotype"])["total_accessibility"].median().unstack())

Depth check after library-size normalization:
genotype                                   Ctrl        KO
cell_type                                                
Astrocytes                             651397.1  651397.1
Deep-layer extratelencephalic neurons  651397.1  651397.1
Endothelial cells                      651397.1  651397.1
Layer 2/3 IT neurons                   651397.1  651397.1
Layer 4 sensory neurons                651397.1  651397.1
Layer 5/6 IT neurons                   651397.1  651397.1
Layer 5a IT neurons                    651397.1  651397.1
Layer 5b PT neurons                    651397.1  651397.1
Layer 6a corticothalamic neurons       651397.1  651397.1
Layer 6b neurons                       651397.1  651397.1
Meningeal fibroblasts                  651397.1  651397.1
Microglia                              651397.1  651397.1
Oligodendrocyte precursor cells        651397.1  651397.1
Oligodendrocytes                       651397.1  651397.1
PV+ interneurons          

In [22]:
# acc_df = acc_df_norm.copy()

In [23]:
# MIN_MEAN_ACC = 0.05
# MIN_VAR = 1e-6
# MIN_METACELLS = 10

# results = []
# for ct in meta["cell_type"].unique():
#     ct_mask = meta["cell_type"] == ct
#     ko_cells = meta.index[ct_mask & (meta["genotype"] == "KO")]
#     ctrl_cells = meta.index[ct_mask & (meta["genotype"] == "Ctrl")]
#     if len(ko_cells) < MIN_METACELLS or len(ctrl_cells) < MIN_METACELLS:
#         print(f"Skipping {ct}: KO={len(ko_cells)}, Ctrl={len(ctrl_cells)} metacells (below {MIN_METACELLS})")
#         continue

#     ko_mean_all = acc_df.loc[ko_cells].mean(axis=0)
#     ctrl_mean_all = acc_df.loc[ctrl_cells].mean(axis=0)
#     var_all = acc_df.loc[list(ko_cells) + list(ctrl_cells)].var(axis=0)

#     keep = ((ko_mean_all > MIN_MEAN_ACC) | (ctrl_mean_all > MIN_MEAN_ACC)) & (var_all > MIN_VAR)
#     var_regions = acc_df.columns[keep]
#     if len(var_regions) == 0:
#         continue

#     ko_mat = acc_df.loc[ko_cells, var_regions].values
#     ctrl_mat = acc_df.loc[ctrl_cells, var_regions].values

#     log2fc = np.log2(ko_mat.mean(axis=0) + 1e-3) - np.log2(ctrl_mat.mean(axis=0) + 1e-3)

#     # Effect size: rank-biserial correlation derived from Mann-Whitney U (0=no effect, 1=complete separation)
#     n1, n2 = ko_mat.shape[0], ctrl_mat.shape[0]
#     pvals = np.ones(len(var_regions))
#     effect_size = np.zeros(len(var_regions))
#     for i in range(len(var_regions)):
#         try:
#             u_stat, p = mannwhitneyu(ko_mat[:, i], ctrl_mat[:, i], alternative="two-sided")
#             pvals[i] = p
#             effect_size[i] = (2 * u_stat / (n1 * n2)) - 1  # rank-biserial correlation, range -1 to 1
#         except ValueError:
#             pvals[i] = 1.0
#             effect_size[i] = 0.0

#     _, padj, _, _ = multipletests(pvals, method="fdr_bh")
#     res = pd.DataFrame({"region": var_regions, "log2FC": log2fc, "effect_size": effect_size,
#                          "pval": pvals, "padj": padj, "cell_type": ct,
#                          "n_KO_metacells": n1, "n_Ctrl_metacells": n2})
#     results.append(res)

# diff_acc = pd.concat(results, ignore_index=True)
# diff_acc["padj_clipped"] = diff_acc["padj"].clip(lower=1e-30)
# diff_acc["log2FC_clipped"] = diff_acc["log2FC"].clip(-10, 10)

# # Primary criterion is now effect size, padj is secondary/exploratory
# EFFECT_SIZE_THRESH = 0.3
# diff_acc["hit"] = (diff_acc["effect_size"].abs() > EFFECT_SIZE_THRESH) & (diff_acc["padj"] < 0.05)

# diff_acc.to_csv(os.path.join(FIG_OUT, "differential_accessibility_KOvsCtrl_allregions.csv"), index=False)
# print(f"Final table: {diff_acc.shape[0]} rows, {diff_acc['hit'].sum()} pass effect_size>|{EFFECT_SIZE_THRESH}| AND padj<0.05")

# cell_types_present = diff_acc["cell_type"].unique()
# fig, axes = plt.subplots(1, len(cell_types_present), figsize=(4.5 * len(cell_types_present), 4.5))
# if len(cell_types_present) == 1:
#     axes = [axes]
# for ax, ct in zip(axes, cell_types_present):
#     sub = diff_acc[diff_acc["cell_type"] == ct]
#     hits = sub[sub["hit"]]
#     ns = sub[~sub["hit"]]
#     ct_color = atac_colors.get(ct, "#4d4d4d")

#     ax.scatter(ns["log2FC_clipped"], ns["effect_size"], s=3, color="lightgrey", rasterized=True, alpha=0.5)
#     ax.scatter(hits["log2FC_clipped"], hits["effect_size"], s=6, color=ct_color, rasterized=True)
#     ax.axhline(EFFECT_SIZE_THRESH, color="grey", linestyle="--", linewidth=0.8)
#     ax.axhline(-EFFECT_SIZE_THRESH, color="grey", linestyle="--", linewidth=0.8)
#     ax.set_ylim(-1.05, 1.05)
#     ax.set_xlim(-10.5, 10.5)
#     n_ko = sub["n_KO_metacells"].iloc[0] if len(sub) else 0
#     n_ctrl = sub["n_Ctrl_metacells"].iloc[0] if len(sub) else 0
#     ax.set_title(f"{ct}\n(KO n={n_ko}, Ctrl n={n_ctrl} metacells)", fontsize=9, color=ct_color)
#     ax.set_xlabel("log2FC (KO vs Ctrl), clipped +/-10")
#     ax.set_ylabel("Rank-biserial effect size")
# plt.tight_layout()
# outpath = os.path.join(FIG_OUT, "Fig4a_differential_accessibility_effectsize_allregions.pdf")
# fig.savefig(outpath, bbox_inches="tight", dpi=300)
# plt.close(fig)
# print("Saved:", outpath)

Skipping Endothelial cells: KO=22, Ctrl=4 metacells (below 10)
Skipping Layer 2/3 IT neurons: KO=18, Ctrl=4 metacells (below 10)
Skipping Layer 5b PT neurons: KO=30, Ctrl=8 metacells (below 10)
Skipping Layer 6b neurons: KO=12, Ctrl=2 metacells (below 10)
Skipping Meningeal fibroblasts: KO=26, Ctrl=8 metacells (below 10)
Skipping Oligodendrocyte precursor cells: KO=38, Ctrl=8 metacells (below 10)
Final table: 4806928 rows, 2516949 pass effect_size>|0.3| AND padj<0.05
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4a_differential_accessibility_effectsize_allregions.pdf


In [24]:
# import os

# GTF_PATH = os.path.expanduser("~/Downloads/mm10.refGene.gtf")

# gtf_cols = ["Chromosome", "source", "feature", "Start", "End", "score", "Strand", "frame", "attributes"]
# gtf = pd.read_csv(GTF_PATH, sep="\t", comment="#", names=gtf_cols, low_memory=False)
# gtf = gtf[gtf["feature"] == "transcript"].copy()

# def extract_gene_name(attr_str):
#     for field in attr_str.split(";"):
#         field = field.strip()
#         if field.startswith("gene_name"):
#             return field.split(" ")[1].strip('"')
#     return None

# gtf["gene_name"] = gtf["attributes"].apply(extract_gene_name)
# gtf = gtf.dropna(subset=["gene_name"])

# gtf["TSS"] = np.where(gtf["Strand"] == "+", gtf["Start"], gtf["End"])
# gene_annot = gtf.drop_duplicates("gene_name")[["gene_name", "Chromosome", "TSS"]]
# print(f"Loaded TSS coordinates for {gene_annot.shape[0]} genes")

# r2g = scplus_obj.uns["Region_to_gene"].copy()
# target_col = "target" if "target" in r2g.columns else "Gene"

# coords = r2g["Region"].str.extract(r"(?P<Chromosome>chr[\w]+)[:\-](?P<Start>\d+)-(?P<End>\d+)")
# r2g["Chromosome"] = coords["Chromosome"]
# r2g["Start"] = coords["Start"].astype(int)
# r2g["End"] = coords["End"].astype(int)

# tss_map = gene_annot.set_index("gene_name")[["Chromosome", "TSS"]]
# r2g = r2g.merge(tss_map, left_on=target_col, right_index=True, how="left", suffixes=("", "_tss"))
# before_n = r2g.shape[0]
# r2g = r2g.dropna(subset=["TSS"])
# r2g = r2g[r2g["Chromosome"] == r2g["Chromosome_tss"]]
# print(f"Region-gene pairs retained after TSS match + chromosome check: {r2g.shape[0]} / {before_n}")

# r2g["region_mid"] = (r2g["Start"] + r2g["End"]) / 2
# r2g["dist_to_TSS"] = (r2g["region_mid"] - r2g["TSS"]).astype(int)

# PROMOTER_DIST_BP = 2000
# r2g["region_class"] = np.where(r2g["dist_to_TSS"].abs() <= PROMOTER_DIST_BP, "Promoter", "Enhancer")

# print(r2g["region_class"].value_counts())

# region_class_map = (
#     r2g.assign(abs_dist=r2g["dist_to_TSS"].abs())
#        .sort_values("abs_dist")
#        .drop_duplicates("Region")
#        .set_index("Region")["region_class"]
#        .to_dict()
# )
# annotated_regions = set(region_class_map.keys())
# print(f"Unique regions classified: {len(annotated_regions)} / {acc_df.shape[1]} total peaks")
# print(pd.Series(region_class_map).value_counts())

Loaded TSS coordinates for 25239 genes
Region-gene pairs retained after TSS match + chromosome check: 3177480 / 4585095
Enhancer    3139787
Promoter      37693
Name: region_class, dtype: int64
Unique regions classified: 697653 / 746029 total peaks
Enhancer    663190
Promoter     34463
dtype: int64


In [25]:
# diff_acc_annot = diff_acc[diff_acc["region"].isin(annotated_regions)].copy()
# diff_acc_annot["region_class"] = diff_acc_annot["region"].map(region_class_map)

# print(f"Annotated subset: {diff_acc_annot.shape[0]} / {diff_acc.shape[0]} region-tests retained")
# print(diff_acc_annot.groupby(["cell_type", "region_class"])["hit"].sum().unstack(fill_value=0))

# diff_acc_annot.to_csv(os.path.join(FIG_OUT, "differential_accessibility_KOvsCtrl_enhancer_promoter.csv"), index=False)

# cell_types_present = diff_acc_annot["cell_type"].unique()
# fig, axes = plt.subplots(1, len(cell_types_present), figsize=(4.5 * len(cell_types_present), 4.5))
# if len(cell_types_present) == 1:
#     axes = [axes]

Annotated subset: 4582567 / 4806928 region-tests retained
region_class                           Enhancer  Promoter
cell_type                                                
Astrocytes                               134926      8984
Deep-layer extratelencephalic neurons    176377      9078
Layer 4 sensory neurons                  258560     17618
Layer 5/6 IT neurons                     144594      8807
Layer 5a IT neurons                      240703     16437
Layer 6a corticothalamic neurons         257470     16592
Microglia                                197885     12649
Oligodendrocytes                         160300     12054
PV+ interneurons                         246881     14137
SST+ interneurons                        256522     16087
VIP+ interneurons                        199724     10663


In [26]:
# class_colors = {"Promoter": "#1b9e77", "Enhancer": "#d95f02"}
# for ax, ct in zip(axes, cell_types_present):
#     sub = diff_acc_annot[diff_acc_annot["cell_type"] == ct]
#     for rc, grp in sub.groupby("region_class"):
#         hits = grp[grp["hit"]]
#         ns = grp[~grp["hit"]]
#         ax.scatter(ns["log2FC_clipped"], ns["effect_size"], s=3, color=class_colors.get(rc, "grey"),
#                    alpha=0.15, rasterized=True, label=None)
#         ax.scatter(hits["log2FC_clipped"], hits["effect_size"], s=8, color=class_colors.get(rc, "grey"),
#                    alpha=0.9, rasterized=True, label=f"{rc} (n hits={len(hits)})")
#     ax.axhline(EFFECT_SIZE_THRESH, color="grey", linestyle="--", linewidth=0.8)
#     ax.axhline(-EFFECT_SIZE_THRESH, color="grey", linestyle="--", linewidth=0.8)
#     ax.set_ylim(-1.05, 1.05)
#     ax.set_xlim(-10.5, 10.5)
#     ax.set_title(ct, fontsize=9, color=atac_colors.get(ct, "#4d4d4d"))
#     ax.set_xlabel("log2FC (KO vs Ctrl)")
#     ax.set_ylabel("Rank-biserial effect size")
#     ax.legend(fontsize=6, loc="upper right", frameon=False)
# plt.tight_layout()
# outpath = os.path.join(FIG_OUT, "Fig4b_differential_accessibility_enhancer_promoter.pdf")
# fig.savefig(outpath, bbox_inches="tight", dpi=300)
# plt.close(fig)
# print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4b_differential_accessibility_enhancer_promoter.pdf


In [18]:
# class_colors = {"Promoter": "#1b9e77", "Enhancer": "#d95f02"}
# from matplotlib.colors import LinearSegmentedColormap

# bg_cmaps = {
#     "Promoter": LinearSegmentedColormap.from_list("prom_bg", ["#ffffff", "#1b9e77"]),
#     "Enhancer": LinearSegmentedColormap.from_list("enh_bg", ["#ffffff", "#d95f02"]),
# }

# fig, axes = plt.subplots(1, len(cell_types_present), figsize=(4.5 * len(cell_types_present), 4.5))
# if len(cell_types_present) == 1:
#     axes = [axes]

# for ax, ct in zip(axes, cell_types_present):
#     sub = diff_acc_annot[diff_acc_annot["cell_type"] == ct]
#     # Background density: only Enhancer non-hits (dominant class) shown as hexbin for clarity
#     bg = sub[(~sub["hit"]) & (sub["region_class"] == "Enhancer")]
#     if len(bg) > 0:
#         ax.hexbin(bg["log2FC_clipped"], bg["effect_size"], gridsize=60,
#                    cmap=bg_cmaps["Enhancer"], mincnt=1, linewidths=0, alpha=0.6, zorder=1)

#     for rc, grp in sub.groupby("region_class"):
#         hits = grp[grp["hit"]]
#         ax.scatter(hits["log2FC_clipped"], hits["effect_size"], s=6, color=class_colors.get(rc, "grey"),
#                    alpha=0.85, rasterized=True, label=f"{rc} (n hits={len(hits)})",
#                    marker="o", edgecolors="black", linewidths=0.2, zorder=2)

#     ax.axhline(EFFECT_SIZE_THRESH, color="grey", linestyle="--", linewidth=0.8, zorder=3)
#     ax.axhline(-EFFECT_SIZE_THRESH, color="grey", linestyle="--", linewidth=0.8, zorder=3)
#     ax.set_ylim(-1.05, 1.05)
#     ax.set_xlim(-10.5, 10.5)
#     ax.set_title(ct, fontsize=9, color=atac_colors.get(ct, "#4d4d4d"))
#     ax.set_xlabel("log2FC (KO vs Ctrl)")
#     ax.set_ylabel("Rank-biserial effect size")
#     ax.legend(fontsize=6, loc="upper right", frameon=False, markerscale=1.5)

# plt.tight_layout()
# outpath = os.path.join(FIG_OUT, "Fig4b_differential_accessibility_enhancer_promoter.pdf")
# fig.savefig(outpath, bbox_inches="tight", dpi=300)
# plt.close(fig)
# print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig4b_differential_accessibility_enhancer_promoter.pdf


In [8]:
# pseudobulk = auc_gene.join(meta, how="inner")
# mean_auc = pseudobulk.groupby(["cell_type", "genotype"]).mean(numeric_only=True)

# top_eregulons = rss_df.max(axis=1).sort_values(ascending=False).head(30).index
# heat_data = mean_auc[top_eregulons].T
# heat_data.columns = [f"{ct}_{geno}" for ct, geno in heat_data.columns]

# fig, ax = plt.subplots(figsize=(max(8, heat_data.shape[1] * 0.9), max(6, heat_data.shape[0] * 0.25)))
# sns.heatmap(heat_data, cmap="RdYlBu_r", center=heat_data.values.mean(),
#             cbar_kws={"label": "Mean eRegulon AUC"}, linewidths=0.3, linecolor="white", ax=ax)
# ax.set_title("Top 30 eRegulons — Mean AUC per Cell Type x Genotype", fontsize=11)

# tick_labels = ax.get_xticklabels()
# for lbl in tick_labels:
#     ct_name = lbl.get_text().rsplit("_", 1)[0]
#     lbl.set_color(atac_colors.get(ct_name, "#000000"))
# ax.set_xticklabels(tick_labels, rotation=45, ha="right")

# plt.tight_layout()
# outpath = os.path.join(FIG_OUT, "Fig5_top_eRegulon_activity_heatmap.pdf")
# fig.savefig(outpath, bbox_inches="tight")
# plt.close(fig)

# mean_auc.reset_index().to_csv(os.path.join(FIG_OUT, "mean_eRegulon_AUC_celltype_genotype.csv"), index=False)
# print("Saved:", outpath)

Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/Fig5_top_eRegulon_activity_heatmap.pdf


In [26]:
# # =============================================================================
# # One-time export: full region -> gene mapping, all regions
# # (Region_to_gene table has columns: Gene, Region, importance, rho)
# # =============================================================================

# import os
# import pickle
# import pandas as pd

# BASE = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus"
# SCPLUS_PKL = os.path.join(BASE, "scplus_obj_geno_with_auc_june_final_REPAIRED.pkl")
# FURTHER_OUT = os.path.join(BASE, "scenicplus_further_analysis")
# FIG_OUT = os.path.join(FURTHER_OUT, "manuscript_figures")

# with open(SCPLUS_PKL, "rb") as f:
#     scplus_obj = pickle.load(f)

# r2g = scplus_obj.uns["Region_to_gene"].copy()
# print(f"Raw Region_to_gene table: {r2g.shape}, columns: {r2g.columns.tolist()}")

# # No TSS-distance column here -- rank by SCENIC+ importance score instead,
# # keep the single best (highest-importance) gene link per region
# region_to_gene_nearest = (
#     r2g.sort_values("importance", ascending=False)
#     .drop_duplicates(subset="Region")
#     .rename(columns={"Region": "region", "Gene": "gene"})
#     [["region", "gene", "importance", "rho"]]
#     .reset_index(drop=True)
# )

# out_path = os.path.join(FIG_OUT, "region_to_gene_nearest_FULL.csv")
# region_to_gene_nearest.to_csv(out_path, index=False)
# print(f"Saved: {out_path}")
# print(f"Total unique regions mapped: {region_to_gene_nearest.shape[0]}")
# print(f"Unique genes: {region_to_gene_nearest['gene'].nunique()}")

Raw Region_to_gene table: (4585095, 4), columns: ['Gene', 'Region', 'importance', 'rho']
Saved: /Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scenicplus/scenicplus_further_analysis/manuscript_figures/region_to_gene_nearest_FULL.csv
Total unique regions mapped: 727348
Unique genes: 21752
